# TP — Modèles de langage basés sur les N-grammes

**Traitement Automatique du Langage Naturel (NLP)** — Master, ISI
Année académique 2026–2027

---

## Objectif

Construire pas à pas un modèle de langage statistique à partir d'un corpus, puis
l'utiliser pour cinq tâches NLP : prédiction du mot suivant, génération de texte,
évaluation de phrases, comparaison de phrases et correction contextuelle.

Chaîne de traitement :

`Corpus → Tokenisation → N-grammes → Comptage → Probabilités → Modèle → Applications`

## Organisation du code

Les fonctions sont définies dans le module **`modele_langage.py`** et importées ici.
Le notebook sert à exécuter, observer et commenter ; il ne redéfinit rien.

**Aucune bibliothèque NLP n'est utilisée** : tout est construit avec les structures
de base de Python (dictionnaires, listes, `Counter`).

In [83]:

%load_ext autoreload
%autoreload 2

from modele_langage import *
from collections import Counter

CORPUS_PATH = "data/corpus.txt"

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


---

## Partie 1 — Prétraitement du corpus

Le corpus brut est une suite de phrases en français. Un programme ne sait pas
manipuler du texte : il faut le transformer en une **structure discrète**.

Quatre opérations :

1. **Mise en minuscules** — sans elle, `Le` et `le` seraient deux entrées
   distinctes du vocabulaire, et les comptages seraient dispersés.
2. **Suppression de la ponctuation** — le point de `poisson.` ferait de ce mot un
   token différent de `poisson`.
3. **Tokenisation** — découpage en unités d'analyse.
4. **Ajout des marqueurs `<s>` et `</s>`** — sans eux, le modèle ne saurait ni
   quels mots peuvent *commencer* une phrase, ni quand une phrase se *termine*.
   Ces deux marqueurs sont traités comme des tokens à part entière.

In [84]:
# Question 1 : afficher les tokens de chaque phrase 
corpus = charger_corpus(CORPUS_PATH)

for i, phrase in enumerate(corpus, 1):
    print(f"Phrase {i} ({len(phrase)} tokens) : {phrase}")

Phrase 1 (7 tokens) : ['<s>', 'le', 'chat', 'mange', 'du', 'poisson', '</s>']
Phrase 2 (7 tokens) : ['<s>', 'le', 'chat', 'aime', 'le', 'poisson', '</s>']
Phrase 3 (8 tokens) : ['<s>', 'le', 'chien', 'mange', 'de', 'la', 'viande', '</s>']
Phrase 4 (7 tokens) : ['<s>', 'le', 'chien', 'aime', 'la', 'viande', '</s>']
Phrase 5 (8 tokens) : ['<s>', 'le', 'chat', 'joue', 'dans', 'le', 'jardin', '</s>']
Phrase 6 (8 tokens) : ['<s>', 'le', 'chien', 'joue', 'dans', 'le', 'jardin', '</s>']


In [85]:
#  Questions 2 et 3 : vocabulaire et sa taille 
vocabulaire = construire_vocabulaire(corpus)
V = len(vocabulaire)

print(f"Vocabulaire ({V} types) :\n")
for mot in vocabulaire:
    print(f"  {mot}")

print(f"\nTaille du vocabulaire : V = {V}")
print(f"  dont 2 marqueurs -> {V - 2} mots réels")

Vocabulaire (15 types) :

  </s>
  <s>
  aime
  chat
  chien
  dans
  de
  du
  jardin
  joue
  la
  le
  mange
  poisson
  viande

Taille du vocabulaire : V = 15
  dont 2 marqueurs -> 13 mots réels


In [86]:
#  Questions 4 et 5 : nombre de tokens et fréquences 
tokens = aplatir(corpus)
N = len(tokens)
frequences = Counter(tokens)

print(f"Nombre total de tokens (occurrences) : N = {N}")
print(f"  dont marqueurs : {2 * len(corpus)}  |  mots réels : {N - 2 * len(corpus)}")
print(f"\nRatio N/V = {N}/{V} = {N/V:.2f} occurrences par type\n")

print(f"{'token':10s} {'fréquence':>10s}")
print("-" * 21)
for mot, freq in frequences.most_common():
    print(f"{mot:10s} {freq:>10d}")

Nombre total de tokens (occurrences) : N = 45
  dont marqueurs : 12  |  mots réels : 33

Ratio N/V = 45/15 = 3.00 occurrences par type

token       fréquence
---------------------
le                  9
<s>                 6
</s>                6
chat                3
chien               3
mange               2
poisson             2
aime                2
la                  2
viande              2
joue                2
dans                2
jardin              2
du                  1
de                  1


### Réponses — Partie 1

**Q1. Tokens de chaque phrase.** Voir la sortie ci-dessus. Chaque phrase devient une
liste de tokens encadrée par `<s>` et `</s>`. Les phrases font 7 ou 8 tokens.

**Q2. Vocabulaire.** `</s>`, `<s>`, `aime`, `chat`, `chien`, `dans`, `de`, `du`,
`jardin`, `joue`, `la`, `le`, `mange`, `poisson`, `viande`.

**Q3. Taille du vocabulaire : V = 15** (13 mots réels + 2 marqueurs).

> **Choix de modélisation.** Les marqueurs sont inclus dans le vocabulaire. C'est
> cohérent : `</s>` est un token que le modèle doit pouvoir *prédire* (il faut bien
> qu'une phrase générée s'arrête). Ce choix a une conséquence directe en Partie 10,
> où V apparaît au dénominateur du lissage de Laplace.

**Q4. Nombre total de tokens : N = 45** — soit 33 mots réels et 12 marqueurs
(6 phrases × 2).

**Q5. Différence entre vocabulaire et nombre de tokens.**

Le vocabulaire compte les **types** (mots distincts) ; N compte les **occurrences**.
Le mot `le` est **une seule** entrée du vocabulaire mais apparaît **9 fois** dans le
corpus : il pèse 20 % des tokens à lui seul.

D'où N = 45 pour V = 15, soit 3 occurrences par type en moyenne.

Cette distinction est le fondement de tout ce qui suit. Un modèle N-gramme estime des
probabilités **en divisant des occurrences par des occurrences** : plus le ratio N/V
est élevé, plus chaque estimation repose sur des observations nombreuses, donc plus
elle est fiable. Ici le ratio est très faible (3) — c'est pourquoi ce corpus jouet
produira des probabilités extrêmes (0 ou 1) que l'on corrigera par lissage.

Quand le corpus grandit, N croît linéairement tandis que V sature : c'est la **loi de
Heaps**. On ne cesse jamais de rencontrer des mots nouveaux, mais de plus en plus
rarement.

---

## Partie 2 — Construction des N-grammes

Un **N-gramme** est une séquence de N tokens consécutifs. Pour la phrase
`<s> le chat mange du poisson </s>` (7 tokens) :

| Ordre | Nom | Nombre | Exemples |
|---|---|---|---|
| N = 1 | unigramme | 7 | `<s>`, `le`, `chat`, … |
| N = 2 | bigramme | 6 | `(<s>, le)`, `(le, chat)`, … |
| N = 3 | trigramme | 5 | `(<s>, le, chat)`, `(le, chat, mange)`, … |

Pour une phrase de *L* tokens, on obtient **L − N + 1** N-grammes d'ordre N.

Ces N-grammes sont **construits phrase par phrase**, jamais sur le corpus aplati :
un bigramme `(</s>, <s>)` à cheval sur deux phrases n'aurait aucun sens linguistique.

In [87]:
# Construction des trois niveaux de N-grammes
unigrammes  = construire_unigrammes(corpus)
bigrammes   = construire_bigrammes(corpus)
trigrammes  = construire_trigrammes(corpus)

afficher_ngrammes(unigrammes, "UNIGRAMMES")

UNIGRAMMES - 15 distincts, 45 occurrences

N-gramme  freq
--------------
le           9
</s>         6
<s>          6
chat         3
chien        3
aime         2
dans         2
jardin       2
joue         2
la           2
mange        2
poisson      2
viande       2
de           1
du           1


In [88]:
#  Question 1 : tous les bigrammes et leurs fréquences 
afficher_ngrammes(bigrammes, "BIGRAMMES")

BIGRAMMES - 23 distincts, 39 occurrences

N-gramme       freq
-------------------
<s> le            6
le chat           3
le chien          3
dans le           2
jardin </s>       2
joue dans         2
la viande         2
le jardin         2
poisson </s>      2
viande </s>       2
aime la           1
aime le           1
chat aime         1
chat joue         1
chat mange        1
chien aime        1
chien joue        1
chien mange       1
de la             1
du poisson        1
le poisson        1
mange de          1
mange du          1


In [89]:
#  Question 2 : tous les trigrammes et leurs fréquences 
afficher_ngrammes(trigrammes, "TRIGRAMMES")

TRIGRAMMES - 25 distincts, 33 occurrences

N-gramme           freq
-----------------------
<s> le chat           3
<s> le chien          3
dans le jardin        2
joue dans le          2
la viande </s>        2
le jardin </s>        2
aime la viande        1
aime le poisson       1
chat aime le          1
chat joue dans        1
chat mange du         1
chien aime la         1
chien joue dans       1
chien mange de        1
de la viande          1
du poisson </s>       1
le chat aime          1
le chat joue          1
le chat mange         1
le chien aime         1
le chien joue         1
le chien mange        1
le poisson </s>       1
mange de la           1
mange du poisson      1


In [90]:
# Questions 3 et 4 : N-grammes les plus fréquents 
for nom, compteur in [("Bigramme", bigrammes), ("Trigramme", trigrammes)]:
    freq_max = max(compteur.values())
    gagnants = [ng for ng, f in compteur.items() if f == freq_max]
    print(f"{nom} le plus fréquent (freq = {freq_max}) :")
    for ng in sorted(gagnants):
        print(f"    ({', '.join(ng)})")
    print()

# Vérification arithmétique : nb de N-grammes = somme(L_i - N + 1)
print("Vérification des totaux")
for n, compteur in [(1, unigrammes), (2, bigrammes), (3, trigrammes)]:
    attendu = sum(len(p) - n + 1 for p in corpus)
    print(f"  N={n} : {sum(compteur.values())} occurrences "
          f"(attendu {attendu}) — {len(compteur)} distincts")

Bigramme le plus fréquent (freq = 6) :
    (<s>, le)

Trigramme le plus fréquent (freq = 3) :
    (<s>, le, chat)
    (<s>, le, chien)

Vérification des totaux
  N=1 : 45 occurrences (attendu 45) — 15 distincts
  N=2 : 39 occurrences (attendu 39) — 23 distincts
  N=3 : 33 occurrences (attendu 33) — 25 distincts


### Réponses — Partie 2

**Q1 et Q2.** Voir les tableaux ci-dessus : 23 bigrammes distincts pour 39 occurrences,
25 trigrammes distincts pour 33 occurrences.

**Q3. Bigramme le plus fréquent : `(<s>, le)`, fréquence 6.**

Il apparaît dans les 6 phrases : *toutes* commencent par `le`. Le modèle en déduira
`P(le | <s>) = 1` — le seul mot capable d'ouvrir une phrase. C'est vrai dans ce corpus,
faux en français : le corpus est trop petit pour que cette certitude soit légitime.

Si l'on écarte les marqueurs, les vainqueurs sont `(le, chat)` et `(le, chien)`,
fréquence 3 chacun.

**Q4. Trigramme le plus fréquent : `(<s>, le, chat)` et `(<s>, le, chien)`, fréquence 3 —
ex æquo.**

L'égalité n'est pas un hasard : le corpus a été construit symétriquement, trois phrases
sur le chat et trois sur le chien.

### Observation centrale : plus N augmente, plus les comptages s'effondrent

| Ordre | Occurrences | Distincts | Ratio occ./distinct |
|---|---|---|---|
| Unigrammes | 45 | 15 | 3.00 |
| Bigrammes | 39 | 23 | 1.70 |
| Trigrammes | 33 | 25 | 1.32 |

Deux mouvements opposés :

- **Les occurrences diminuent** (45 → 39 → 33), mécaniquement : chaque phrase de *L*
  tokens fournit L − N + 1 N-grammes, donc une de moins à chaque ordre.
- **Les distincts augmentent** (15 → 23 → 25), car il y a combinatoirement plus de
  séquences longues possibles que de mots isolés.

Résultat : le ratio chute de 3.00 à 1.32. **Sur 25 trigrammes, 19 n'apparaissent qu'une
seule fois.** Estimer une probabilité à partir d'une observation unique n'a aucune
robustesse statistique.

C'est le **problème de dispersion des données** (*data sparsity*), et il gouverne tout
le reste du TP : il explique pourquoi les probabilités nulles sont omniprésentes
(Partie 9), pourquoi le lissage est nécessaire (Partie 10), et pourquoi le trigramme
— pourtant plus informé — est plus fragile que le bigramme (Partie 11).

Théoriquement, avec V = 15, il existe 15² = 225 bigrammes possibles ; on n'en observe
que 23, soit **10 %**. Pour les trigrammes : 25 observés sur 3 375 possibles, soit
**0,7 %**.

---

## Partie 3 — Construction d'un modèle bigramme

### Le problème que résout l'hypothèse de Markov

La probabilité exacte d'une phrase se décompose par la **règle de la chaîne** :

$$P(w_1, \dots, w_n) = \prod_{i=1}^{n} P(w_i \mid w_1, \dots, w_{i-1})$$

Cette formule est exacte, mais inutilisable : estimer $P(w_i \mid w_1 \dots w_{i-1})$
exigerait d'avoir observé l'historique complet dans le corpus. Pour un historique de
10 mots et V = 10 000, cela ferait $10^{40}$ contextes distincts.

L'**hypothèse de Markov** tronque l'historique aux N−1 derniers mots. Pour un bigramme :

$$P(w_i \mid w_1, \dots, w_{i-1}) \approx P(w_i \mid w_{i-1})$$

C'est une approximation *fausse* linguistiquement — un mot dépend souvent de mots
lointains — mais elle rend le modèle estimable.

### Estimation par maximum de vraisemblance

$$P(w_i \mid w_{i-1}) = \frac{C(w_{i-1}, w_i)}{C(w_{i-1})}$$

On divise le nombre de fois où le bigramme a été vu par le nombre de fois où le mot
de contexte a été vu. C'est la fréquence relative observée : l'estimateur qui maximise
la vraisemblance du corpus d'entraînement.

**Point de vigilance sur le dénominateur.** On divise par $C(w_{i-1})$, le comptage
**unigramme**, et non par le nombre de bigrammes commençant par $w_{i-1}$. Les deux
coïncident presque, mais pas exactement : `</s>` termine une phrase et n'ouvre jamais
de bigramme. Cette petite différence garantit que les probabilités somment à 1 pour
tout contexte autre que `</s>`.

In [9]:
# Entraînement du modèle bigramme
modele = ModeleNgramme(corpus)

print(f"Modèle entraîné : {len(modele.corpus)} phrases, "
      f"N = {modele.N} tokens, V = {modele.V} types")

Modèle entraîné : 6 phrases, N = 45 tokens, V = 15 types


In [91]:
# Les six probabilités demandées par le TP 
paires = [
    ("le", "chat"), ("le", "chien"),
    ("chat", "mange"), ("chat", "aime"),
    ("du", "poisson"), ("la", "viande"),
]

for precedent, mot in paires:
    print(modele.detail_probabilite(precedent, mot))

P(chat | le) = C(le, chat) / C(le) = 3/9 = 0.3333
P(chien | le) = C(le, chien) / C(le) = 3/9 = 0.3333
P(mange | chat) = C(chat, mange) / C(chat) = 1/3 = 0.3333
P(aime | chat) = C(chat, aime) / C(chat) = 1/3 = 0.3333
P(poisson | du) = C(du, poisson) / C(du) = 1/1 = 1.0000
P(viande | la) = C(la, viande) / C(la) = 2/2 = 1.0000


In [92]:
#  Distributions de probabilité pour quelques contextes 
for contexte in ["<s>", "le", "chat", "chien", "mange", "joue"]:
    distribution = modele.successeurs(contexte)
    total = sum(distribution.values())
    print(f"Après « {contexte} »  (C = {modele.compte_unigramme(contexte)}) :")
    for mot, p in distribution.items():
        barre = "█" * int(p * 30)
        print(f"    P({mot:8s}| {contexte:5s}) = {p:.4f}  {barre}")
    print(f"    → somme = {total:.4f}\n")

Après « <s> »  (C = 6) :
    P(le      | <s>  ) = 1.0000  ██████████████████████████████
    → somme = 1.0000

Après « le »  (C = 9) :
    P(chat    | le   ) = 0.3333  ██████████
    P(chien   | le   ) = 0.3333  ██████████
    P(jardin  | le   ) = 0.2222  ██████
    P(poisson | le   ) = 0.1111  ███
    → somme = 1.0000

Après « chat »  (C = 3) :
    P(aime    | chat ) = 0.3333  ██████████
    P(joue    | chat ) = 0.3333  ██████████
    P(mange   | chat ) = 0.3333  ██████████
    → somme = 1.0000

Après « chien »  (C = 3) :
    P(aime    | chien) = 0.3333  ██████████
    P(joue    | chien) = 0.3333  ██████████
    P(mange   | chien) = 0.3333  ██████████
    → somme = 1.0000

Après « mange »  (C = 2) :
    P(de      | mange) = 0.5000  ███████████████
    P(du      | mange) = 0.5000  ███████████████
    → somme = 1.0000

Après « joue »  (C = 2) :
    P(dans    | joue ) = 1.0000  ██████████████████████████████
    → somme = 1.0000



In [93]:
# Quelques probabilités nulles, pour la Question 1
print("Bigrammes jamais observés :\n")
for precedent, mot in [("chat", "pain"), ("chat", "le"),
                       ("le", "mange"), ("viande", "poisson")]:
    print("   ", modele.detail_probabilite(precedent, mot))

Bigrammes jamais observés :

    P(pain | chat) = C(chat, pain) / C(chat) = 0/3 = 0.0000
    P(le | chat) = C(chat, le) / C(chat) = 0/3 = 0.0000
    P(mange | le) = C(le, mange) / C(le) = 0/9 = 0.0000
    P(poisson | viande) = C(viande, poisson) / C(viande) = 0/2 = 0.0000


### Réponses — Partie 3

**Q1. Pourquoi certaines probabilités sont-elles nulles ?**

Parce que le numérateur $C(w_{i-1}, w_i)$ vaut 0 : le bigramme **n'a jamais été observé
dans le corpus**. Exemple : $P(\text{pain} \mid \text{chat}) = 0/3 = 0$.

Il faut bien distinguer deux causes très différentes, que le modèle confond :

- **L'impossibilité linguistique** — `(viande, poisson)` est effectivement mal formé
  en français.
- **L'absence d'observation** — `(chat, pain)` est parfaitement correct en français,
  mais notre corpus de 6 phrases ne parle pas de pain.

Le modèle ne fait aucune différence entre les deux : il attribue 0 dans les deux cas.
C'est une **erreur d'estimation**, pas une vérité linguistique. Un corpus de 45 tokens
ne peut pas prétendre couvrir le français.

Chiffré : avec V = 15, il existe 225 bigrammes possibles ; 23 sont observés.
**202 bigrammes, soit 90 %, ont une probabilité nulle.**

**Q2. Que signifie une probabilité élevée pour un bigramme ?**

Que le second mot suit très régulièrement le premier **dans ce corpus**. Trois nuances :

- $P(\text{poisson} \mid \text{du}) = 1/1 = 1$ : le maximum absolu, mais fondé sur
  **une seule** observation. Statistiquement, cela ne vaut rien — c'est un artefact de
  la petite taille du corpus, pas une régularité du français.
- $P(\text{viande} \mid \text{la}) = 2/2 = 1$ : même certitude, sur 2 observations.
  À peine mieux.
- $P(\text{chat} \mid \text{le}) = 3/9 = 0.33$ : plus faible, mais reposant sur
  9 observations du contexte — **l'estimation la plus fiable des trois**.

Leçon : une probabilité élevée n'est informative que si le **dénominateur** est grand.
La valeur seule ne dit rien ; il faut regarder sur combien d'observations elle repose.

**Q3. Que signifie une probabilité nulle ?**

Littéralement : « ce bigramme n'a jamais été vu ». Le modèle le traite comme
**impossible**, ce qui est bien plus fort.

La conséquence est grave, car les probabilités de phrase sont des **produits** :

$$P(S) = \prod_i P(w_i \mid w_{i-1})$$

Un seul facteur nul annule tout le produit. Une phrase de 20 mots parfaitement
grammaticale reçoit $P(S) = 0$ à cause d'un seul bigramme jamais rencontré. Le modèle
devient alors incapable de comparer deux phrases — elles sont toutes deux à zéro.

C'est le **problème des comptes nuls**, traité en Partie 9 et résolu par le lissage de
Laplace en Partie 10.

### Ce que révèlent les distributions

Deux formes très différentes cohabitent :

- **Distributions plates** : après `chat`, les trois successeurs `aime`, `joue`, `mange`
  sont à 1/3 chacun. Le contexte n'apporte aucune information discriminante — le modèle
  est indécis.
- **Distributions dégénérées** : après `<s>`, `le` est à 1.0 ; après `du` ou `dans`,
  un seul successeur. Le modèle est *certain*, mais cette certitude vient de la pauvreté
  du corpus, pas d'une régularité de la langue.

Un vrai corpus produit des distributions intermédiaires : un successeur dominant, une
longue traîne de successeurs rares. Ici on n'a que les deux extrêmes.

---

## Partie 4 — Tâche NLP 1 : prédiction du mot suivant

C'est l'application la plus visible d'un modèle de langage : le clavier prédictif
d'un téléphone. L'utilisateur tape un contexte, le modèle propose les mots les plus
probables.

Le principe :

$$\hat{w} = \arg\max_{w \in V} P(w \mid w_{i-1})$$

On parcourt les successeurs observés du dernier mot et on retient celui de plus forte
probabilité.

**Conséquence directe de l'hypothèse de Markov :** dans un modèle bigramme, seul le
**dernier mot** du contexte compte. Prédire après « le chat » et après « chat » donne
strictement le même résultat — le mot `le` est ignoré.

In [94]:
#  Tests demandés par le TP 
for contexte in ["le chat", "le chien", "le", "chat"]:
    modele.afficher_prediction(contexte)

Contexte : « le chat »   -> mot conditionnant : « chat »  (C = 3)
    P(aime    | chat  ) = 0.3333  ##########
    P(joue    | chat  ) = 0.3333  ##########
    P(mange   | chat  ) = 0.3333  ##########
    => EGALITE entre aime, joue, mange -> choix alphabetique : « aime »

Contexte : « le chien »   -> mot conditionnant : « chien »  (C = 3)
    P(aime    | chien ) = 0.3333  ##########
    P(joue    | chien ) = 0.3333  ##########
    P(mange   | chien ) = 0.3333  ##########
    => EGALITE entre aime, joue, mange -> choix alphabetique : « aime »

Contexte : « le »   -> mot conditionnant : « le »  (C = 9)
    P(chat    | le    ) = 0.3333  ##########
    P(chien   | le    ) = 0.3333  ##########
    P(jardin  | le    ) = 0.2222  ######
    P(poisson | le    ) = 0.1111  ###
    => EGALITE entre chat, chien -> choix alphabetique : « chat »

Contexte : « chat »   -> mot conditionnant : « chat »  (C = 3)
    P(aime    | chat  ) = 0.3333  ##########
    P(joue    | chat  ) = 0.3333  ##########
  

In [95]:
#  Deux cas limites intéressants 
for contexte in ["<s>", "jardin", "poisson", "pain"]:
    modele.afficher_prediction(contexte)

Contexte : « <s> »   -> mot conditionnant : « <s> »  (C = 6)
    P(le      | <s>   ) = 1.0000  ##############################
    => mot le plus probable : « le »

Contexte : « jardin »   -> mot conditionnant : « jardin »  (C = 2)
    P(</s>    | jardin) = 1.0000  ##############################
    => mot le plus probable : « </s> »

Contexte : « poisson »   -> mot conditionnant : « poisson »  (C = 2)
    P(</s>    | poisson) = 1.0000  ##############################
    => mot le plus probable : « </s> »

Contexte : « pain »   -> mot conditionnant : « pain »  (C = 0)
    aucun successeur observe : le modele ne peut rien predire



In [96]:
#  Question : pourquoi P(chat | le) ≠ P(le | chat) ? 
print("Les deux directions reposent sur des comptages différents :\n")
print(f"  C(le, chat) = {modele.compte_bigramme('le', 'chat')}   "
      f"C(le)   = {modele.compte_unigramme('le')}")
print(f"  C(chat, le) = {modele.compte_bigramme('chat', 'le')}   "
      f"C(chat) = {modele.compte_unigramme('chat')}\n")
print(modele.detail_probabilite("le", "chat"))
print(modele.detail_probabilite("chat", "le"))

Les deux directions reposent sur des comptages différents :

  C(le, chat) = 3   C(le)   = 9
  C(chat, le) = 0   C(chat) = 3

P(chat | le) = C(le, chat) / C(le) = 3/9 = 0.3333
P(le | chat) = C(chat, le) / C(chat) = 0/3 = 0.0000


### Réponses — Partie 4

**Question : différence entre $P(\text{chat} \mid \text{le})$ et $P(\text{le} \mid \text{chat})$.
Pourquoi ne sont-elles généralement pas égales ?**

Ce sont deux quantités qui ne posent pas la même question :

- $P(\text{chat} \mid \text{le})$ : « sachant que je viens de lire `le`, quelle chance
  que le mot suivant soit `chat` ? » → $C(\text{le}, \text{chat}) / C(\text{le}) = 3/9 = 0.333$
- $P(\text{le} \mid \text{chat})$ : « sachant que je viens de lire `chat`, quelle chance
  que le mot suivant soit `le` ? » → $C(\text{chat}, \text{le}) / C(\text{chat}) = 0/3 = 0$

**Trois raisons de l'asymétrie :**

1. **Les dénominateurs diffèrent.** $C(\text{le}) = 9$ contre $C(\text{chat}) = 3$.
   Même à numérateur identique, les résultats divergeraient.

2. **Les numérateurs diffèrent, car un bigramme est ordonné.** $(le, chat)$ et
   $(chat, le)$ sont deux N-grammes **distincts** : le premier apparaît 3 fois, le
   second jamais. C'est précisément ce qui permet aux N-grammes de capturer l'ordre
   des mots (voir Partie 7).

3. **Linguistiquement, le français est asymétrique.** Un déterminant précède son nom ;
   `le chat` est bien formé, `chat le` ne l'est pas.

Formellement, l'égalité n'aurait lieu que si $C(w_1) = C(w_2)$, par la règle de Bayes :

$$P(A \mid B) = \frac{P(B \mid A) \, P(A)}{P(B)}$$

Les deux ne coïncident que lorsque $P(A) = P(B)$ — cas exceptionnel.

**Confondre les deux est l'erreur la plus fréquente sur ce chapitre.** Un modèle de
langage lit de gauche à droite : c'est toujours le contexte *passé* qui conditionne
le mot *à venir*, jamais l'inverse.

### Deux observations sur les prédictions

**Les égalités massives.** Après `chat`, les trois candidats sont à exactement 1/3.
Le modèle est incapable de trancher, et la « prédiction » n'est qu'une convention de
départage alphabétique. Avec un corpus réaliste, les fréquences se différencient
naturellement et ce cas devient rare.

**Le mot hors vocabulaire.** Après `pain` — absent du corpus — le modèle ne retourne
**rien du tout** : $C(\text{pain}) = 0$, donc aucun successeur, et le calcul serait une
division par zéro. Un modèle de production traite ce cas avec un token spécial `<UNK>`
ou un mécanisme de repli (*backoff*) vers l'unigramme. Ici, le modèle est simplement
muet.

---

## Partie 5 — Tâche NLP 2 : génération de texte

Un modèle de langage ne fait pas qu'*évaluer* du texte : il peut en **produire**.
L'algorithme est celui de la Partie 4, appliqué en boucle :

1. partir de `<s>` ;
2. prédire le mot suivant ;
3. l'ajouter à la phrase ;
4. recommencer jusqu'à `</s>`.

$$w_{i} = \arg\max_{w} P(w \mid w_{i-1})$$

### Le problème de l'argmax

Le TP demande de générer **cinq phrases**. Or l'argmax est **déterministe** : à contexte
identique, il renvoie toujours le même mot. Les cinq phrases seront donc rigoureusement
identiques.

C'est pourquoi on implémente une seconde stratégie, l'**échantillonnage** : au lieu de
prendre le maximum, on **tire au sort** le mot suivant selon sa distribution de
probabilité. Un mot à 0.5 sort une fois sur deux, un mot à 0.1 une fois sur dix.

C'est exactement le rôle de la *température* dans les modèles génératifs modernes :
argmax = température 0 (déterministe, répétitif), échantillonnage = température 1
(varié, parfois incohérent).

In [97]:
#  Stratégie 1 : argmax (celle demandée par le TP) 
print("Génération par argmax — 5 essais :\n")
for i in range(5):
    tokens = modele.generer_phrase(mode="argmax")
    print(f"  {i+1}. {tokens}")

print("\n→ Les 5 phrases sont IDENTIQUES : l'argmax est déterministe.")

Génération par argmax — 5 essais :

  1. ['<s>', 'le', 'chat', 'aime', 'la', 'viande', '</s>']
  2. ['<s>', 'le', 'chat', 'aime', 'la', 'viande', '</s>']
  3. ['<s>', 'le', 'chat', 'aime', 'la', 'viande', '</s>']
  4. ['<s>', 'le', 'chat', 'aime', 'la', 'viande', '</s>']
  5. ['<s>', 'le', 'chat', 'aime', 'la', 'viande', '</s>']

→ Les 5 phrases sont IDENTIQUES : l'argmax est déterministe.


In [98]:
#  Trace détaillée du chemin suivi par l'argmax
print("Chemin de décision :\n")
tokens = modele.generer_phrase(mode="argmax", tracer=True)
print(f"\nPhrase générée : « {modele.phrase_lisible(tokens)} »")

Chemin de décision :

    <s>      -> le       (p = 1.000, 1 candidat(s))
    le       -> chat     (p = 0.333, 4 candidat(s))
    chat     -> aime     (p = 0.333, 3 candidat(s))
    aime     -> la       (p = 0.500, 2 candidat(s))
    la       -> viande   (p = 1.000, 1 candidat(s))
    viande   -> </s>     (p = 1.000, 1 candidat(s))

Phrase générée : « le chat aime la viande »


In [99]:
#  Stratégie 2 : échantillonnage selon la distribution 
from collections import Counter as _C

print("Génération par échantillonnage — 10 phrases (graine fixée) :\n")
phrases = []
for i in range(10):
    tokens = modele.generer_phrase(mode="echantillon", graine=42 if i == 0 else None)
    texte = modele.phrase_lisible(tokens)
    phrases.append(texte)
    print(f"  {i+1:2d}. {texte}")

distinctes = _C(phrases)
print(f"\n→ {len(distinctes)} phrases distinctes sur 10")

Génération par échantillonnage — 10 phrases (graine fixée) :

   1. le chat aime la viande
   2. le chat joue dans le chien aime la viande
   3. le chien mange de la viande
   4. le poisson
   5. le chat mange du poisson
   6. le poisson
   7. le jardin
   8. le chien mange de la viande
   9. le chat aime le chien joue dans le poisson
  10. le chat mange de la viande

→ 8 phrases distinctes sur 10


In [100]:
#  Les phrases générées existent-elles dans le corpus ? 
phrases_corpus = {modele.phrase_lisible(p) for p in corpus}

print("Phrases du corpus d'entraînement :")
for p in sorted(phrases_corpus):
    print(f"  - {p}")

print("\nAnalyse des phrases générées :\n")
for texte in sorted(set(phrases)):
    statut = "DANS le corpus" if texte in phrases_corpus else "NOUVELLE"
    print(f"  [{statut:14s}] {texte}")

Phrases du corpus d'entraînement :
  - le chat aime le poisson
  - le chat joue dans le jardin
  - le chat mange du poisson
  - le chien aime la viande
  - le chien joue dans le jardin
  - le chien mange de la viande

Analyse des phrases générées :

  [NOUVELLE      ] le chat aime la viande
  [NOUVELLE      ] le chat aime le chien joue dans le poisson
  [NOUVELLE      ] le chat joue dans le chien aime la viande
  [NOUVELLE      ] le chat mange de la viande
  [DANS le corpus] le chat mange du poisson
  [DANS le corpus] le chien mange de la viande
  [NOUVELLE      ] le jardin
  [NOUVELLE      ] le poisson


### Réponses — Partie 5

**Question : pourquoi les phrases générées par un modèle bigramme peuvent-elles être
grammaticalement incorrectes ou peu naturelles ?**

Regardons un exemple produit par l'échantillonnage :

> *le chat joue dans le chien aime la viande*

Chaque bigramme, pris isolément, est **parfaitement légitime** — tous ont été observés
dans le corpus :

`(le, chat)` ✓ · `(chat, joue)` ✓ · `(joue, dans)` ✓ · `(dans, le)` ✓ ·
`(le, chien)` ✓ · `(chien, aime)` ✓ · `(aime, la)` ✓ · `(la, viande)` ✓

La phrase est pourtant absurde. **Quatre raisons :**

**1. Mémoire d'un seul mot.** Arrivé à `dans le ___`, le modèle a déjà oublié qu'il
avait écrit `joue`. Il ne sait pas qu'une phrase est en cours ni qu'un sujet a déjà été
posé. Chaque décision est prise dans l'ignorance totale de ce qui précède au-delà d'un
mot.

**2. Aucune notion de structure syntaxique.** Le modèle ignore ce qu'est un sujet, un
verbe, un complément. Il ne peut pas savoir qu'une phrase ne prend pas deux sujets.
Il n'y a **aucune grammaire** dans un modèle N-gramme, seulement des comptages.

**3. Aucune sémantique.** Rien n'empêche `le chat aime la viande` — grammaticalement
irréprochable, jamais vu dans le corpus (le corpus associe la viande au chien). Le
modèle ignore ce que sont un chat et de la viande.

**4. La localité se propage.** La validité est garantie **par paires**, jamais
globalement. Enchaîner des transitions individuellement correctes ne produit pas une
phrase correcte — c'est la limite structurelle de l'hypothèse de Markov.

### La contrepartie : le modèle généralise

Un point qui mérite d'être souligné : **`le chat aime la viande` n'est dans aucune
phrase du corpus.** Le corpus associe le chat au poisson et le chien à la viande. Le
modèle a recombiné des fragments pour produire une phrase **nouvelle et pourtant
correcte**.

C'est la propriété fondamentale d'un modèle génératif : il ne récite pas son corpus,
il en recombine les régularités. La capacité de produire du neuf et la tendance à
produire de l'incohérent sont **le même mécanisme** — on ne peut pas avoir l'une sans
l'autre.

### Argmax ou échantillonnage ?

| | Argmax | Échantillonnage |
|---|---|---|
| Déterminisme | oui — 1 seule phrase possible | non — variété |
| Cohérence | maximale | dégradée |
| Couverture du corpus | un unique chemin | tout le graphe |
| Analogue moderne | température = 0 | température = 1 |

L'argmax bloque le modèle sur **un seul chemin** dans le graphe des transitions : il
n'atteindra jamais `poisson`, alors que le corpus en parle deux fois. Le TP demandant
cinq phrases, seul l'échantillonnage rend l'exercice sensé.

C'est le même dilemme qu'aujourd'hui avec les LLM : une température basse donne des
réponses fiables mais ternes, une température haute donne de la créativité et des
hallucinations.

---

## Partie 6 — Tâche NLP 3 : probabilité d'une phrase

On applique la règle de la chaîne, avec l'approximation bigramme :

$$P(S) = P(w_1 \mid \texttt{<s>}) \times \prod_{i=2}^{n} P(w_i \mid w_{i-1}) \times P(\texttt{</s>} \mid w_n)$$

Deux points de méthode :

**Le marqueur `<s>` fournit le premier facteur.** Sans lui, on ne saurait pas évaluer
la probabilité que la phrase *commence* par ce mot-là.

**Le marqueur `</s>` fournit le dernier.** Sans lui, une phrase tronquée comme
« le chat mange du » recevrait la même probabilité que la phrase complète. C'est `</s>`
qui pénalise les phrases qui s'arrêtent au mauvais endroit.

Pour une phrase de *n* mots, on multiplie donc **n + 1** facteurs.

In [101]:
#  Les trois phrases demandées par le TP 
phrases_test = [
    "le chat mange du poisson",
    "le chien mange de la viande",
    "le chat joue dans le jardin",
]

for phrase in phrases_test:
    modele.probabilite_phrase(phrase, tracer=True)

P(le chat mange du poisson) =
    P(le       | <s>     ) = 1.000000
    P(chat     | le      ) = 0.333333
    P(mange    | chat    ) = 0.333333
    P(du       | mange   ) = 0.500000
    P(poisson  | du      ) = 1.000000
    P(</s>     | poisson ) = 1.000000
              ---------------------------
    produit = 0.05555556

P(le chien mange de la viande) =
    P(le       | <s>     ) = 1.000000
    P(chien    | le      ) = 0.333333
    P(mange    | chien   ) = 0.333333
    P(de       | mange   ) = 0.500000
    P(la       | de      ) = 1.000000
    P(viande   | la      ) = 1.000000
    P(</s>     | viande  ) = 1.000000
              ---------------------------
    produit = 0.05555556

P(le chat joue dans le jardin) =
    P(le       | <s>     ) = 1.000000
    P(chat     | le      ) = 0.333333
    P(joue     | chat    ) = 0.333333
    P(dans     | joue    ) = 1.000000
    P(le       | dans    ) = 1.000000
    P(jardin   | le      ) = 0.222222
    P(</s>     | jardin  ) = 1.000000
        

In [102]:
#  Récapitulatif : probabilité, log-probabilité, longueur
print(f"{'phrase':32s} {'nb fact.':>8s} {'P(S)':>12s} {'log2 P(S)':>11s}")
print("-" * 67)
for phrase in phrases_test:
    tokens = tokeniser(phrase)
    p = modele.probabilite_phrase(phrase)
    lp = modele.log_probabilite_phrase(phrase)
    print(f"{phrase:32s} {len(tokens)-1:>8d} {p:>12.6f} {lp:>11.4f}")

phrase                           nb fact.         P(S)   log2 P(S)
-------------------------------------------------------------------
le chat mange du poisson                6     0.055556     -4.1699
le chien mange de la viande             7     0.055556     -4.1699
le chat joue dans le jardin             7     0.024691     -5.3399


In [103]:
#  Phrases absentes du corpus d'entraînement
print("Le modèle sait-il évaluer des phrases qu'il n'a jamais vues ?\n")

for phrase in ["le chat aime la viande",       # recombinaison valide
               "le chat mange de la viande",   # recombinaison valide
               "le chat mange du pain"]:       # contient un mot inconnu
    p = modele.probabilite_phrase(phrase)
    lp = modele.log_probabilite_phrase(phrase)
    verdict = "évaluable" if p > 0 else "ANNULÉE (bigramme jamais vu)"
    print(f"  {phrase:30s} P = {p:.6f}   log2 = {lp:>9.4f}   {verdict}")

Le modèle sait-il évaluer des phrases qu'il n'a jamais vues ?

  le chat aime la viande         P = 0.055556   log2 =   -4.1699   évaluable
  le chat mange de la viande     P = 0.055556   log2 =   -4.1699   évaluable
  le chat mange du pain          P = 0.000000   log2 =      -inf   ANNULÉE (bigramme jamais vu)


### Réponses — Partie 6

**Question : que signifie une probabilité élevée pour une phrase ?**

Que la phrase est **typique du corpus d'entraînement** : ses transitions entre mots
correspondent à celles que le modèle a fréquemment observées. « Typique » n'est pas
« correcte » — c'est une notion statistique, pas grammaticale. Le modèle mesure la
conformité aux régularités qu'il a vues, rien d'autre.

Trois précisions importantes :

**1. Les valeurs absolues n'ont aucun sens en soi.** P = 0.0556 n'est ni « bon » ni
« mauvais ». Ces nombres ne sont interprétables que **comparés entre eux**, sur des
phrases de longueur voisine.

**2. La probabilité décroît mécaniquement avec la longueur.** Chaque facteur est ≤ 1,
donc chaque mot supplémentaire ne peut que faire baisser P(S). Une phrase longue et
parfaitement naturelle aura toujours une probabilité plus basse qu'une phrase courte
et médiocre. **Comparer directement deux phrases de longueurs différentes n'a pas de
sens.** C'est pour cela qu'on utilise la perplexité, qui normalise par le nombre de
transitions.

**3. Un exemple frappant :** les phrases 1 et 2 ont exactement la même probabilité
(0.0556) malgré des longueurs différentes (6 et 7 facteurs). La phrase 2 est plus
longue mais ses facteurs supplémentaires valent 1.0 — le corpus ne laisse aucun choix
après `de` et après `la`. Une transition certaine ne coûte rien.

**Pourquoi la phrase 3 est-elle moins probable ?** Le seul facteur qui la distingue est
$P(\text{jardin} \mid \text{le}) = 2/9 = 0.222$, contre $0.333$ pour `chat` ou `chien`.
Le mot `jardin` suit `le` moins souvent que les animaux : la phrase est donc moins
typique du corpus.

### Ce que révèle l'évaluation hors corpus

**« le chat aime la viande » obtient P = 0.0556** — la même valeur que les phrases
d'entraînement, alors qu'elle n'apparaît **nulle part** dans le corpus. Le modèle
**généralise** : il évalue des phrases nouvelles à partir de fragments observés.
C'est ce qui distingue un modèle d'une table de correspondance.

**« le chat mange du pain » obtient P = 0** : le mot `pain` est absent du vocabulaire,
donc $C(\text{du}, \text{pain}) = 0$, et un seul facteur nul annule tout le produit.

C'est le **problème des comptes nuls** dans sa forme la plus concrète. Notez ce que
cela implique : le modèle ne peut pas dire que « le chat mange du pain » est *plus*
plausible que « pain du mange chat le » — les deux sont à 0, donc rigoureusement
indistinguables. Le modèle a perdu tout pouvoir de discrimination. Partie 9 et Partie 10.

### Pourquoi passer aux logarithmes

Sur ce corpus, P(S) reste lisible. Sur un vrai texte, non : une phrase de 20 mots dont
chaque transition vaut $10^{-3}$ donne $P(S) = 10^{-60}$. Au-delà d'environ 300 facteurs,
un `float64` arrondit à 0.0 — c'est le **soupassement numérique** (*underflow*), et le
résultat devient faux sans aucun message d'erreur.

La solution standard :

$$\log_2 P(S) = \sum_{i} \log_2 P(w_i \mid w_{i-1})$$

Le produit devient une somme, les valeurs restent dans une plage exploitable
(−4.17 au lieu de 0.0556), et le logarithme étant strictement croissant, **l'ordre des
phrases est préservé** : comparer des log-probabilités revient exactement à comparer
des probabilités. C'est ce que fait tout système réel.

---

## Partie 7 — Tâche NLP 4 : comparaison de phrases

$$S_1 = \text{« le chat mange du poisson »} \qquad S_2 = \text{« poisson le mange chat du »}$$

Les deux phrases contiennent **exactement les mêmes cinq mots**, dans un ordre différent.
S₂ est une permutation de S₁.

Un modèle « sac de mots » (*bag of words*) — TF-IDF, Naive Bayes classique — leur
attribuerait des représentations **identiques**, car il ne conserve que les fréquences
de mots, jamais leur position.

Un modèle N-gramme, lui, doit pouvoir les distinguer. C'est exactement ce que teste
cette partie.

In [104]:
#  Vérification préalable : mêmes mots, ordre différent 
from collections import Counter as _C

S1 = "le chat mange du poisson"
S2 = "poisson le mange chat du"

mots1 = _C(tokeniser(S1, ajouter_marqueurs=False))
mots2 = _C(tokeniser(S2, ajouter_marqueurs=False))

print(f"S1 : {sorted(mots1.elements())}")
print(f"S2 : {sorted(mots2.elements())}")
print(f"\nMêmes mots ? {mots1 == mots2}")
print("→ Un modèle « sac de mots » ne pourrait PAS les distinguer.")

S1 : ['chat', 'du', 'le', 'mange', 'poisson']
S2 : ['chat', 'du', 'le', 'mange', 'poisson']

Mêmes mots ? True
→ Un modèle « sac de mots » ne pourrait PAS les distinguer.


In [105]:
# Calcul et comparaison 
p1, p2 = modele.comparer_phrases(S1, S2)

S1 = « le chat mange du poisson »
P(le chat mange du poisson) =
    P(le       | <s>     ) = 1.000000
    P(chat     | le      ) = 0.333333
    P(mange    | chat    ) = 0.333333
    P(du       | mange   ) = 0.500000
    P(poisson  | du      ) = 1.000000
    P(</s>     | poisson ) = 1.000000
              ---------------------------
    produit = 0.05555556

S2 = « poisson le mange chat du »
P(poisson le mange chat du) =
    P(poisson  | <s>     ) = 0.000000  <-- ZERO
    P(le       | poisson ) = 0.000000  <-- ZERO
    P(mange    | le      ) = 0.000000  <-- ZERO
    P(chat     | mange   ) = 0.000000  <-- ZERO
    P(du       | chat    ) = 0.000000  <-- ZERO
    P(</s>     | du      ) = 0.000000  <-- ZERO
              ---------------------------
    produit = 0.00000000

VERDICT : S1 est plus probable que S2 (infini).


In [106]:
# Détail : pourquoi TOUS les bigrammes de S2 sont-ils nuls ? 
tokens = tokeniser(S2)

print("Bigrammes de S2 et leurs comptages :\n")
print(f"{'bigramme':24s} {'C':>3s} {'P':>8s}")
print("-" * 37)
for precedent, mot in zip(tokens, tokens[1:]):
    c = modele.compte_bigramme(precedent, mot)
    p = modele.probabilite_bigramme(precedent, mot)
    print(f"({precedent}, {mot})".ljust(24) + f"{c:>3d} {p:>8.4f}")

nuls = sum(1 for a, b in zip(tokens, tokens[1:])
           if modele.compte_bigramme(a, b) == 0)
print(f"\n→ {nuls} bigrammes nuls sur {len(tokens)-1}")

Bigrammes de S2 et leurs comptages :

bigramme                   C        P
-------------------------------------
(<s>, poisson)            0   0.0000
(poisson, le)             0   0.0000
(le, mange)               0   0.0000
(mange, chat)             0   0.0000
(chat, du)                0   0.0000
(du, </s>)                0   0.0000

→ 6 bigrammes nuls sur 6


In [107]:
#  Toutes les permutations ne se valent pas 
print("Le modèle distingue-t-il les degrés de désordre ?\n")
for phrase in ["le chat mange du poisson",   # ordre correct
               "le poisson mange du chat",   # syntaxe correcte, sens inversé
               "chat le mange poisson du",    # désordre partiel
               "poisson le mange chat du"]:   # désordre total
    p = modele.probabilite_phrase(phrase)
    print(f"  {phrase:28s} P = {p:.6f}")

Le modèle distingue-t-il les degrés de désordre ?

  le chat mange du poisson     P = 0.055556
  le poisson mange du chat     P = 0.000000
  chat le mange poisson du     P = 0.000000
  poisson le mange chat du     P = 0.000000


### Réponse — Partie 7

**Question : comment les N-grammes permettent-ils de tenir compte de l'ordre des mots ?**

**Parce qu'un N-gramme est un tuple ordonné, pas un ensemble.**

`(le, chat)` et `(chat, le)` sont deux clés **différentes** dans le dictionnaire de
comptages. La première a été vue 3 fois, la seconde jamais. En stockant des *séquences*
plutôt que des *mots isolés*, le modèle enregistre l'information de position.

Trois niveaux de lecture :

**1. Au niveau du comptage.** Chaque bigramme observé est un fait attesté sur l'ordre :
« dans ce corpus, `chat` suit `le` ». Permuter les mots produit des paires que le corpus
n'a jamais vues.

**2. Au niveau de la probabilité conditionnelle.** $P(w_i \mid w_{i-1})$ est
**asymétrique** par construction (Partie 4) : elle encode une direction de lecture. Une
probabilité conditionnelle *est* une contrainte d'ordre.

**3. Au niveau de la phrase.** P(S) est un **produit** de ces contraintes. Permuter les
mots casse chaque contrainte simultanément — d'où l'effondrement total observé : 6 zéros
sur 6.

### La comparaison avec le sac de mots

Ce résultat est la démonstration la plus nette de l'apport des N-grammes :

| | Sac de mots | Bigrammes |
|---|---|---|
| Représentation de S₁ | {le:1, chat:1, mange:1, du:1, poisson:1} | 6 transitions ordonnées |
| Représentation de S₂ | **identique** | 6 transitions, toutes différentes |
| Distinction possible ? | **non** | **oui** |

Le sac de mots donne le même vecteur aux deux phrases : « le chat mange le poisson » et
« le poisson mange le chat » y sont indiscernables, alors que le sens est inversé.

### Une limite qu'il faut voir

Le verdict de la Partie 7 est net, mais **fragile** — parce qu'il repose sur des zéros.

Les tests supplémentaires le montrent : « le poisson mange du chat » (syntaxe française
correcte, sens absurde) et « poisson le mange chat du » (désordre total) reçoivent
**la même probabilité : 0**. Le modèle ne les distingue pas.

Cela veut dire que le modèle ne mesure pas *à quel point* une phrase est désordonnée.
Il ne dit pas « S₂ est moins probable » mais « S₂ est impossible » — et il dit
la même chose de toute phrase contenant un seul bigramme jamais vu, y compris
une phrase parfaitement correcte comme « le chat mange du pain » (Partie 6).

Le modèle a raison ici, mais **par accident** : il ne classe pas, il annule. Après
lissage de Laplace (Partie 10), on recalculera cette comparaison et l'on obtiendra
un vrai **classement gradué** : S₁ restera plus probable que S₂, mais avec un rapport
fini et interprétable, et les degrés de désordre deviendront distinguables.

---

## Partie 8 — Tâche NLP 5 : correction contextuelle

### Ce qu'un correcteur lexical sait faire, et ce qu'il ne sait pas

Un correcteur fondé sur un **dictionnaire** vérifie une seule chose : le mot existe-t-il ?

- `connexoin` → absent du dictionnaire → **faute détectée** ✓
- `Il a cet ans.` → `cet` existe, `ans` existe → **aucune faute détectée** ✗

Or la phrase est fautive. L'erreur n'est pas **lexicale** mais **contextuelle** : chaque
mot est valide isolément, c'est leur *combinaison* qui ne l'est pas.

C'est précisément ce qu'un modèle N-gramme peut détecter, puisqu'il modélise des
séquences et non des mots isolés.

### Corpus d'entraînement dédié

```
Il a sept ans.        Il a cet objet.
Elle a sept ans.      Elle a cet objet.
Mon frère a sept ans. Il prend cet objet.
```

Corpus délibérément équilibré : `sept` et `cet` y apparaissent tous les deux après `a`.
Le mot précédent seul ne peut donc pas trancher — il faudra regarder à droite.

In [108]:
#  Entraînement d'un SECOND modèle sur le corpus de correction 
corpus_corr = charger_corpus("data/corpus_correction.txt")
modele_corr = ModeleNgramme(corpus_corr)

print(f"Corpus de correction : {len(corpus_corr)} phrases, "
      f"N = {modele_corr.N}, V = {modele_corr.V}")
print(f"Vocabulaire : {modele_corr.vocabulaire}\n")

for phrase in corpus_corr:
    print(f"  {phrase}")

Corpus de correction : 6 phrases, N = 37, V = 12
Vocabulaire : ['</s>', '<s>', 'a', 'ans', 'cet', 'elle', 'frère', 'il', 'mon', 'objet', 'prend', 'sept']

  ['<s>', 'il', 'a', 'sept', 'ans', '</s>']
  ['<s>', 'elle', 'a', 'sept', 'ans', '</s>']
  ['<s>', 'mon', 'frère', 'a', 'sept', 'ans', '</s>']
  ['<s>', 'il', 'a', 'cet', 'objet', '</s>']
  ['<s>', 'elle', 'a', 'cet', 'objet', '</s>']
  ['<s>', 'il', 'prend', 'cet', 'objet', '</s>']


In [28]:
# --- Comparaison demandée : P(sept | a) vs P(cet | a) ---
print("Contexte GAUCHE seul :\n")
print("   ", modele_corr.detail_probabilite("a", "sept"))
print("   ", modele_corr.detail_probabilite("a", "cet"))

print("\nContexte DROIT :\n")
print("   ", modele_corr.detail_probabilite("sept", "ans"))
print("   ", modele_corr.detail_probabilite("cet", "ans"))

Contexte GAUCHE seul :



    P(sept | a) = C(a, sept) / C(a) = 3/5 = 0.6000
    P(cet | a) = C(a, cet) / C(a) = 2/5 = 0.4000

Contexte DROIT :

    P(ans | sept) = C(sept, ans) / C(sept) = 3/3 = 1.0000
    P(ans | cet) = C(cet, ans) / C(cet) = 0/3 = 0.0000


In [109]:
# Correction de « Il a cet ans. » 
phrase_fautive = "Il a cet ans."
print(f"Phrase soumise : « {phrase_fautive} »\n")

corrige = corriger_phrase(modele_corr, phrase_fautive)
print(f"Phrase corrigée : « {modele_corr.phrase_lisible(corrige)} »")

Phrase soumise : « Il a cet ans. »

Position 2 : « a »  (contexte : il ___ cet)
    a      : P(a|il) = 0.6667  x  P(cet|a) = 0.4000  =  0.2667 <-- retenu
    à      : P(à|il) = 0.0000  x  P(cet|à) = 0.0000  =  0.0000
    => aucun changement

Position 3 : « cet »  (contexte : a ___ ans)
    cet    : P(cet|a) = 0.4000  x  P(ans|cet) = 0.0000  =  0.0000
    sept   : P(sept|a) = 0.6000  x  P(ans|sept) = 1.0000  =  0.6000 <-- retenu
    => CORRECTION : « cet » remplace par « sept »

Phrase corrigée : « il a sept ans »


In [110]:
# Test inverse : le système corrige-t-il dans l'autre sens ? 
print("Phrase soumise : « Il a sept objet. »\n")
corrige2 = corriger_phrase(modele_corr, "Il a sept objet.")
print(f"Phrase corrigée : « {modele_corr.phrase_lisible(corrige2)} »")

Phrase soumise : « Il a sept objet. »

Position 2 : « a »  (contexte : il ___ sept)
    a      : P(a|il) = 0.6667  x  P(sept|a) = 0.6000  =  0.4000 <-- retenu
    à      : P(à|il) = 0.0000  x  P(sept|à) = 0.0000  =  0.0000
    => aucun changement

Position 3 : « sept »  (contexte : a ___ objet)
    cet    : P(cet|a) = 0.4000  x  P(objet|cet) = 1.0000  =  0.4000 <-- retenu
    sept   : P(sept|a) = 0.6000  x  P(objet|sept) = 0.0000  =  0.0000
    => CORRECTION : « sept » remplace par « cet »

Phrase corrigée : « il a cet objet »


In [111]:
#  Vérification par la probabilité de phrase complète 
print("Le verdict est-il confirmé au niveau de la phrase entière ?\n")
for phrase in ["il a cet ans", "il a sept ans",
               "il a cet objet", "il a sept objet"]:
    p = modele_corr.probabilite_phrase(phrase)
    statut = "plausible" if p > 0 else "IMPOSSIBLE"
    print(f"  {phrase:20s} P = {p:.6f}   {statut}")

Le verdict est-il confirmé au niveau de la phrase entière ?

  il a cet ans         P = 0.000000   IMPOSSIBLE
  il a sept ans        P = 0.200000   plausible
  il a cet objet       P = 0.133333   plausible
  il a sept objet      P = 0.000000   IMPOSSIBLE


### Réponse — Partie 8

**Question : pourquoi les N-grammes permettent-ils de traiter des erreurs que la simple
vérification dans un dictionnaire ne peut pas détecter ?**

Parce que les deux approches ne modélisent pas le même objet.

| | Dictionnaire | Modèle N-gramme |
|---|---|---|
| Unité modélisée | le mot isolé | la séquence de mots |
| Question posée | « ce mot existe-t-il ? » | « cette suite est-elle attendue ici ? » |
| Réponse | binaire (oui/non) | graduée (une probabilité) |
| Détecte `connexoin` | oui | oui (mot hors vocabulaire) |
| Détecte `cet ans` | **non** | **oui** |

Un dictionnaire est une **liste** : il ne connaît que l'appartenance. Il ne contient
aucune information sur ce qui peut suivre quoi. Face à `cet ans`, les deux mots sont
dans la liste, donc rien à signaler.

Un modèle N-gramme est une **distribution sur les transitions**. Il a observé que `ans`
suit `sept` (3 fois) et jamais `cet`. Le bigramme `(cet, ans)` a une probabilité nulle :
l'anomalie est mesurable.

Autrement dit, **le dictionnaire valide les mots, le modèle N-gramme valide les
enchaînements**. Les fautes contextuelles — homophones, accords, confusions
grammaticales — vivent précisément dans les enchaînements.

### Le contexte gauche ne suffit pas

Le résultat le plus instructif de cette partie : **la comparaison demandée par le TP
donne la bonne réponse pour la mauvaise raison.**

$P(\text{sept} \mid \text{a}) = 0.6$ contre $P(\text{cet} \mid \text{a}) = 0.4$ : `sept`
l'emporte, mais l'écart est faible et ne reflète qu'un déséquilibre du corpus — trois
phrases avec `sept` contre deux avec `cet` après `a`. Si le corpus avait contenu une
phrase de plus avec `cet`, le verdict se serait inversé. **La décision serait un
artefact d'échantillonnage.**

Ce qui tranche réellement, c'est le mot **suivant** : $P(\text{ans} \mid \text{cet}) = 0$
contre $P(\text{ans} \mid \text{sept}) = 1$. Le score bidirectionnel

$$\text{score}(c) = P(c \mid w_{i-1}) \times P(w_{i+1} \mid c)$$

donne 0.6 contre 0.0 — un écart net et robuste. C'est pourquoi l'implémentation regarde
**des deux côtés** du mot suspect, ce que le TP ne demandait pas explicitement.

| Candidat | $P(c \mid a)$ | × | $P(\text{ans} \mid c)$ | = | score |
|---|---|---|---|---|---|
| `cet` | 0.4000 | | 0.0000 | | **0.0000** |
| `sept` | 0.6000 | | 1.0000 | | **0.6000** |

Le test symétrique le confirme : devant `objet`, le même système corrige
`sept → cet`. Il n'a aucune préférence intrinsèque pour l'un ou l'autre mot ;
**seul le contexte décide**. C'est la définition même d'un correcteur contextuel.

### Limites de ce correcteur

Trois faiblesses à connaître :

**1. Il dépend d'une liste de confusions prédéfinie.** Un vrai correcteur génère les
candidats automatiquement, par **distance d'édition** (Levenshtein ≤ 2) ou par
similarité phonétique. Ici, `CONFUSIONS` est écrite à la main.

**2. Il repose sur des zéros.** La décision vient d'une probabilité nulle, donc du
manque de données. Sur un corpus réel, `(cet, ans)` finirait par apparaître dans une
phrase mal écrite, et le score ne serait plus nul. Un correcteur robuste travaille sur
des **probabilités lissées** et un **seuil de décision** : ne corriger que si l'écart
dépasse une marge, pour éviter de modifier du texte correct.

**3. Sa portée est d'un mot.** Il ne peut pas détecter un accord à distance
(« les enfants que j'ai vu » → « vus »), où le mot déclencheur est cinq positions
en amont. Il faudrait des N-grammes d'ordre élevé — impraticable — ou un modèle
neuronal, ce qui nous amène au défi final du TP.

---

## Partie 9 — Le problème des probabilités nulles

Si le corpus ne contient jamais le bigramme `(chat, pain)`, alors :

$$C(\text{chat}, \text{pain}) = 0 \quad \Longrightarrow \quad P(\text{pain} \mid \text{chat}) = \frac{0}{C(\text{chat})} = 0$$

C'est le **problème des comptes nuls** (*zero-count problem*). Il ne s'agit pas d'un bug
d'implémentation mais d'une **limite structurelle** de l'estimation par maximum de
vraisemblance : le MLE attribue toute la masse de probabilité aux événements observés,
et exactement zéro à tous les autres.

> ⚠️ Une seule probabilité nulle rend la probabilité de la phrase entière égale à zéro,
> puisque P(S) est un **produit**.

In [112]:
#  La matrice des comptages de bigrammes 
print("C(w1, w2) — lignes = mot précédent, colonnes = mot suivant\n")
matrice_bigrammes(modele)
print("\n(un point = comptage nul)")

C(w1, w2) — lignes = mot précédent, colonnes = mot suivant

           </s>    <s>   aime   chat  chien   dans     de     du jardin   joue     la     le  mange poisso viande
</s>          .      .      .      .      .      .      .      .      .      .      .      .      .      .      .
<s>           .      .      .      .      .      .      .      .      .      .      .      6      .      .      .
aime          .      .      .      .      .      .      .      .      .      .      1      1      .      .      .
chat          .      .      1      .      .      .      .      .      .      1      .      .      1      .      .
chien         .      .      1      .      .      .      .      .      .      1      .      .      1      .      .
dans          .      .      .      .      .      .      .      .      .      .      .      2      .      .      .
de            .      .      .      .      .      .      .      .      .      .      1      .      .      .      .
du            .      .      

In [113]:
#  Quantifier le phénomène 
stats = couverture_bigrammes(modele)

print(f"Taille du vocabulaire        V = {stats['V']}")
print(f"Bigrammes théoriquement possibles  V² = {stats['possibles']}")
print(f"Bigrammes observés                      {stats['observes']}")
print(f"Bigrammes de comptage NUL               {stats['nuls']}")
print(f"\nTaux de couverture : {stats['taux_couverture']:.1%}")
print(f"→ {1 - stats['taux_couverture']:.1%} de la matrice est vide")
print(f"\nBigrammes vus UNE SEULE fois (hapax) : "
      f"{stats['hapax']} sur {stats['observes']}")

Taille du vocabulaire        V = 15
Bigrammes théoriquement possibles  V² = 225
Bigrammes observés                      23
Bigrammes de comptage NUL               202

Taux de couverture : 10.2%
→ 89.8% de la matrice est vide

Bigrammes vus UNE SEULE fois (hapax) : 13 sur 23


In [114]:
#  Travail demandé : trouver des bigrammes de fréquence nulle 
nuls = bigrammes_nuls(modele, exclure_marqueurs=True)
print(f"{len(nuls)} bigrammes nuls (marqueurs exclus). Quelques exemples :\n")

# Cas linguistiquement corrects, mais jamais observés
interessants = [("chat", "la"), ("chien", "du"), ("chat", "de"),
                ("chien", "le"), ("le", "viande"), ("aime", "du")]
for w1, w2 in interessants:
    print(f"  C({w1}, {w2}) = {modele.compte_bigramme(w1, w2)}  "
          f"→ P({w2} | {w1}) = {modele.probabilite_bigramme(w1, w2):.1f}")

150 bigrammes nuls (marqueurs exclus). Quelques exemples :

  C(chat, la) = 0  → P(la | chat) = 0.0
  C(chien, du) = 0  → P(du | chien) = 0.0
  C(chat, de) = 0  → P(de | chat) = 0.0
  C(chien, le) = 0  → P(le | chien) = 0.0
  C(le, viande) = 0  → P(viande | le) = 0.0
  C(aime, du) = 0  → P(du | aime) = 0.0


In [115]:
#  La conséquence : une phrase correcte annulée 
print("Effet d'un seul zéro sur une phrase entière :\n")
modele.probabilite_phrase("le chien aime du poisson", tracer=True)

Effet d'un seul zéro sur une phrase entière :

P(le chien aime du poisson) =
    P(le       | <s>     ) = 1.000000
    P(chien    | le      ) = 0.333333
    P(aime     | chien   ) = 0.333333
    P(du       | aime    ) = 0.000000  <-- ZERO
    P(poisson  | du      ) = 1.000000
    P(</s>     | poisson ) = 1.000000
              ---------------------------
    produit = 0.00000000



0.0

In [116]:
#  Le problème s'aggrave avec l'ordre du modèle 
print(f"{'ordre':10s} {'distincts':>10s} {'possibles':>12s} "
      f"{'couverture':>11s} {'hapax':>7s}")
print("-" * 54)
for n, compteur in [(1, modele.unigrammes), (2, modele.bigrammes),
                    (3, modele.trigrammes)]:
    possibles = modele.V ** n
    hapax = sum(1 for f in compteur.values() if f == 1)
    print(f"{n}-gramme  {len(compteur):>10d} {possibles:>12d} "
          f"{len(compteur)/possibles:>10.2%} "
          f"{hapax:>4d}/{len(compteur)}")

ordre       distincts    possibles  couverture   hapax
------------------------------------------------------
1-gramme          15           15    100.00%    2/15
2-gramme          23          225     10.22%   13/23
3-gramme          25         3375      0.74%   19/25


### Réponses — Partie 9

**Travail demandé : trouver des bigrammes de fréquence nulle et expliquer le problème.**

**202 bigrammes sur 225 ont un comptage nul**, soit près de 90 % de la matrice.
Quelques exemples parlants :

| Bigramme | C | Statut linguistique |
|---|---|---|
| `(chat, la)` | 0 | **correct** — « le chat la regarde » |
| `(chien, du)` | 0 | **correct** — « le chien du voisin » |
| `(aime, du)` | 0 | **correct** — « il aime du poisson » |
| `(le, viande)` | 0 | incorrect (accord) |
| `(viande, poisson)` | 0 | incorrect |

C'est là le cœur du problème : **le modèle attribue exactement 0 aux séquences
impossibles et aux séquences simplement non observées.** Il confond « je n'ai jamais
vu ça » avec « ça ne peut pas exister ».

### Pourquoi cela pose problème

**1. Un seul zéro annule tout.** P(S) est un produit. Testons
« le chien aime du poisson » — phrase parfaitement correcte en français :

| Facteur | Valeur |
|---|---|
| P(le \| `<s>`) | 1.000 |
| P(chien \| le) | 0.333 |
| P(aime \| chien) | 0.333 |
| **P(du \| aime)** | **0.000** ← |
| P(poisson \| du) | 1.000 |
| P(`</s>` \| poisson) | 1.000 |
| **Produit** | **0.000** |

Cinq facteurs sur six sont bons. Le sixième détruit tout.

**2. Le modèle perd son pouvoir de discrimination.** Rappelons la Partie 7 :
« le chat mange du pain » (correcte) et « poisson le mange chat du » (absurde)
reçoivent **la même valeur : 0**. Comparer devient impossible. Un modèle qui répond
« zéro » à presque tout ne classe plus rien.

**3. C'est mathématiquement incohérent.** Le MLE prétend que la probabilité totale des
séquences non vues est nulle. Or on sait qu'elle ne l'est pas : le français produit
sans cesse des combinaisons nouvelles. Le modèle est **sur-confiant** — il donne 100 %
de la masse à 10 % de l'espace.

**4. Ce n'est pas un défaut du petit corpus.** L'intuition dit « avec plus de données,
le problème disparaît ». C'est faux, et c'est un résultat classique : par la **loi de
Zipf**, la fréquence d'un mot est inversement proportionnelle à son rang. Quelle que
soit la taille du corpus, une grande partie du vocabulaire reste rare, et l'immense
majorité des bigrammes possibles reste non observée. Sur un corpus d'un milliard de
mots avec V = 100 000, il y a 10¹⁰ bigrammes possibles : la matrice reste vide à plus
de 99 %.

**Le problème n'est donc pas un manque de données : il est structurel.** Il faut une
solution mathématique — c'est le rôle du lissage.

### Le signal d'alarme : les hapax

**13 des 23 bigrammes observés n'apparaissent qu'une seule fois** (ce sont les *hapax*).
Pour les trigrammes, 19 sur 25.

C'est le symptôme le plus inquiétant. Ces bigrammes reçoivent une probabilité non nulle
— mais fondée sur **une observation unique**. Le modèle traite avec la même assurance :

- $P(\text{chat} \mid \text{le}) = 3/9$, appuyée sur 9 observations du contexte ;
- $P(\text{poisson} \mid \text{du}) = 1/1 = 1$, appuyée sur **une seule**.

La frontière entre « vu une fois » (probabilité maximale) et « jamais vu »
(probabilité nulle) est **arbitraire**. Une observation supplémentaire ou manquante
fait basculer d'un extrême à l'autre.

D'où l'idée du lissage : **transférer un peu de masse de probabilité des événements
observés vers les événements non observés**, pour lisser cette falaise. C'est la
Partie 10.

---

## Partie 10 — Lissage de Laplace

L'idée est simple : **faire comme si chaque bigramme avait été vu une fois de plus
qu'en réalité.**

$$P_{\text{Laplace}}(w_i \mid w_{i-1}) = \frac{C(w_{i-1}, w_i) + 1}{C(w_{i-1}) + V}$$

On ajoute 1 à chaque numérateur, et V au dénominateur. Le lissage *add-one* est le plus
simple de sa famille ; on l'appelle aussi lissage de Laplace, par référence à sa
« règle de succession ».

### Pourquoi V au dénominateur, et pas 1 ?

Une distribution de probabilité doit sommer à 1. Si on ajoute 1 à chacun des V
numérateurs possibles, on a ajouté V au total :

$$\sum_{w \in V} \big(C(w_{i-1}, w) + 1\big) = C(w_{i-1}) + V$$

Le dénominateur doit donc absorber exactement V pour préserver la normalisation.
Ajouter 1 seulement au numérateur casserait la distribution.

### Version généralisée

$$P_{\alpha}(w_i \mid w_{i-1}) = \frac{C(w_{i-1}, w_i) + \alpha}{C(w_{i-1}) + \alpha V}$$

Avec α = 1 on retrouve Laplace. Avec α < 1 (typiquement 0.1 ou 0.01), on parle de
**lissage de Lidstone** ou *add-α* : plus doux, il déforme moins les comptages observés.
L'implémentation prend α en paramètre.

In [117]:
#  Le cas emblématique : P(pain | chat) 
print("Avant lissage (MLE) :")
print("   ", modele.detail_probabilite("chat", "pain"))
print("\nAprès lissage de Laplace :")
print("   ", modele.detail_laplace("chat", "pain"))
print(f"\n→ La probabilité n'est plus nulle : "
      f"{modele.probabilite_laplace('chat', 'pain'):.6f}")

Avant lissage (MLE) :
    P(pain | chat) = C(chat, pain) / C(chat) = 0/3 = 0.0000

Après lissage de Laplace :
    P_Laplace(pain | chat) = (0 + 1) / (3 + 1x15) = 0.055556

→ La probabilité n'est plus nulle : 0.055556


In [118]:
# Comparaison MLE / Laplace sur un contexte fréquent
contexte = "le"
print(f"Contexte « {contexte} » — C({contexte}) = "
      f"{modele.compte_unigramme(contexte)}\n")
print(f"{'mot':10s} {'C':>3s} {'MLE':>8s} {'Laplace':>9s} {'C*':>7s} {'écart':>8s}")
print("-" * 50)

for mot in modele.vocabulaire:
    c = modele.compte_bigramme(contexte, mot)
    mle = modele.probabilite_bigramme(contexte, mot)
    lap = modele.probabilite_laplace(contexte, mot)
    creconst = modele.compte_reconstitue(contexte, mot)
    print(f"{mot:10s} {c:>3d} {mle:>8.4f} {lap:>9.4f} {creconst:>7.2f} "
          f"{lap - mle:>+8.4f}")

d = modele.distribution_laplace(contexte)
print(f"\nSomme de la distribution lissée : {sum(d.values()):.10f}")

Contexte « le » — C(le) = 9

mot          C      MLE   Laplace      C*    écart
--------------------------------------------------
</s>         0   0.0000    0.0417    0.38  +0.0417
<s>          0   0.0000    0.0417    0.38  +0.0417
aime         0   0.0000    0.0417    0.38  +0.0417
chat         3   0.3333    0.1667    1.50  -0.1667
chien        3   0.3333    0.1667    1.50  -0.1667
dans         0   0.0000    0.0417    0.38  +0.0417
de           0   0.0000    0.0417    0.38  +0.0417
du           0   0.0000    0.0417    0.38  +0.0417
jardin       2   0.2222    0.1250    1.12  -0.0972
joue         0   0.0000    0.0417    0.38  +0.0417
la           0   0.0000    0.0417    0.38  +0.0417
le           0   0.0000    0.0417    0.38  +0.0417
mange        0   0.0000    0.0417    0.38  +0.0417
poisson      1   0.1111    0.0833    0.75  -0.0278
viande       0   0.0000    0.0417    0.38  +0.0417

Somme de la distribution lissée : 1.0000000000


In [ ]:
#  L'effet dépend fortement de la fréquence du contexte 
print("Contexte FRÉQUENT vs contexte RARE\n")
for contexte, mot in [("le", "chat"), ("du", "poisson")]:
    c_ctx = modele.compte_unigramme(contexte)
    mle = modele.probabilite_bigramme(contexte, mot)
    lap = modele.probabilite_laplace(contexte, mot)
    print(f"  C({contexte}) = {c_ctx:2d}   "
          f"P({mot} | {contexte}) : MLE = {mle:.4f} → Laplace = {lap:.4f}   "
          f"(perte {100*(1-lap/mle):.0f} %)")

Contexte FRÉQUENT vs contexte RARE

  C(le) =  9   P(chat | le) : MLE = 0.3333 → Laplace = 0.1667   (perte 50 %)
  C(du) =  1   P(poisson | du) : MLE = 1.0000 → Laplace = 0.1250   (perte 88 %)


In [119]:
#  Reprise de la Partie 7 : la comparaison devient graduée 
print("Comparaison S1 / S2 AVEC lissage :\n")
for phrase in ["le chat mange du poisson",   # phrase du corpus
               "le chat mange du pain",       # correcte, mot inconnu
               "le poisson mange du chat",    # syntaxe OK, sens absurde
               "poisson le mange chat du"]:   # désordre total
    mle = modele.probabilite_phrase(phrase)
    lap = modele.probabilite_phrase(phrase, lissage=True)
    print(f"  {phrase:28s} MLE = {mle:.6f}   Laplace = {lap:.3e}")

p1 = modele.probabilite_phrase("le chat mange du poisson", lissage=True)
p2 = modele.probabilite_phrase("poisson le mange chat du", lissage=True)
print(f"\n→ S1 est {p1/p2:.0f} fois plus probable que S2 "
      f"(rapport fini, donc interprétable)")

Comparaison S1 / S2 AVEC lissage :

  le chat mange du poisson     MLE = 0.055556   Laplace = 1.602e-05
  le chat mange du pain        MLE = 0.000000   Laplace = 3.026e-06
  le poisson mange du chat     MLE = 0.000000   Laplace = 6.675e-07
  poisson le mange chat du     MLE = 0.000000   Laplace = 2.384e-08

→ S1 est 672 fois plus probable que S2 (rapport fini, donc interprétable)


In [120]:
# Influence du paramètre alpha
print(f"{'alpha':>7s} {'P(chat|le)':>12s} {'P(pain|chat)':>14s}")
print("-" * 35)
for alpha in [1.0, 0.5, 0.1, 0.01, 0.001]:
    print(f"{alpha:>7g} {modele.probabilite_laplace('le','chat',alpha):>12.4f} "
          f"{modele.probabilite_laplace('chat','pain',alpha):>14.6f}")
print(f"{'MLE':>7s} {modele.probabilite_bigramme('le','chat'):>12.4f} "
      f"{modele.probabilite_bigramme('chat','pain'):>14.6f}")

  alpha   P(chat|le)   P(pain|chat)
-----------------------------------
      1       0.1667       0.055556
    0.5       0.2121       0.047619
    0.1       0.2952       0.022222
   0.01       0.3290       0.003175
  0.001       0.3329       0.000332
    MLE       0.3333       0.000000


### Réponses — Partie 10

**Q1. Pourquoi ajoute-t-on 1 au numérateur ?**

Pour qu'aucun comptage ne soit jamais nul. On fait comme si **chaque bigramme du
vocabulaire avait été observé une fois de plus** qu'en réalité. Un bigramme jamais vu
passe de 0 à 1, donc de probabilité nulle à probabilité faible mais **strictement
positive**.

L'interprétation est bayésienne : le +1 correspond à un **a priori uniforme** sur les
bigrammes. Avant d'observer le corpus, on suppose toutes les transitions également
possibles ; les données viennent ensuite corriger cet a priori. C'est la « règle de
succession » de Laplace : ne jamais conclure à l'impossibilité à partir d'une simple
absence d'observation.

**Q2. Pourquoi ajoute-t-on V au dénominateur ?**

Pour préserver la **normalisation**. Une distribution de probabilité doit sommer à 1 :

$$\sum_{w \in V} P(w \mid w_{i-1}) = 1$$

Le vocabulaire compte V mots ; on ajoute 1 à chacun des V numérateurs, donc **V au
total**. Le dénominateur doit absorber exactement cette quantité :

$$\sum_{w \in V} \big(C(w_{i-1}, w) + 1\big) = C(w_{i-1}) + V$$

La cellule ci-dessus le vérifie numériquement : la distribution lissée somme à
1.0000000000.

C'est aussi la raison pour laquelle **le choix d'inclure `<s>` et `</s>` dans le
vocabulaire (Partie 1) a des conséquences ici** : V = 15 et non 13, donc le dénominateur
change et toutes les probabilités lissées avec lui.

**Q3. Pourquoi le dénominateur doit-il être modifié ?**

Sans cette correction, le lissage produirait des valeurs qui **ne sont plus des
probabilités**. Vérifions sur le contexte `le` (C = 9, V = 15) :

- Avec le bon dénominateur : $\sum_w \frac{C+1}{9+15} = \frac{9+15}{24} = 1$ ✓
- Avec l'ancien dénominateur : $\sum_w \frac{C+1}{9} = \frac{24}{9} = 2.67$ ✗

On obtiendrait une « probabilité » totale de 267 %. Tous les calculs de phrase — qui
sont des produits de ces valeurs — deviendraient des nombres arbitraires, impossibles
à comparer entre contextes de fréquences différentes.

**Modifier le numérateur sans modifier le dénominateur est l'erreur la plus courante
sur cette formule.**

### Comparaison P(pain | chat) avant / après

| | Formule | Valeur |
|---|---|---|
| MLE | 0 / 3 | **0.000000** |
| Laplace | (0+1) / (3+15) | **0.055556** |

Le bigramme n'est plus impossible, seulement improbable. C'est exactement ce qu'on
voulait : `chat pain` est une séquence que le français autorise, notre corpus ne
l'avait simplement jamais rencontrée.

### Ce que le lissage coûte

Le lissage ne crée pas de probabilité — il en **redistribue**. La masse donnée aux
bigrammes inconnus est prise sur les bigrammes observés.

$P(\text{chat} \mid \text{le})$ tombe de 0.3333 à 0.1667 : **la moitié**. Et la colonne
C\* montre où va cette masse : le bigramme `(le, chat)`, vu 3 fois, se comporte désormais
comme s'il n'avait été vu que **1.5 fois**.

**L'effet dépend violemment de la fréquence du contexte :**

| Contexte | C | MLE | Laplace | Perte |
|---|---|---|---|---|
| `le` (fréquent) | 9 | 0.3333 | 0.1667 | 50 % |
| `du` (rare) | 1 | 1.0000 | 0.1250 | **87 %** |

C'est le **défaut majeur du lissage add-one** : quand $C(w_{i-1}) \ll V$, le terme V
écrase le comptage réel. Ici C(du) = 1 contre V = 15 — le dénominateur est presque
entièrement constitué de masse fictive, et l'observation réelle ne pèse quasiment plus
rien.

Sur un corpus réaliste, le problème est bien pire : avec V = 50 000, un contexte vu
10 fois a un dénominateur de 50 010. Le lissage add-one **détruit** l'information
observée. C'est pourquoi il n'est plus utilisé en production, remplacé par des méthodes
qui redistribuent plus intelligemment : **Good-Turing** (estime la masse des inconnus
à partir du nombre d'hapax), **backoff de Katz** (recule vers l'unigramme quand le
bigramme est absent), **interpolation de Jelinek-Mercer**, ou **Kneser-Ney**, longtemps
l'état de l'art.

Le paramètre α permet d'atténuer le problème : avec α = 0.01, P(chat | le) remonte à
0.3288, très proche du MLE, tout en gardant P(pain | chat) = 0.0031 > 0. Le compromis
se règle sur un corpus de validation.

### Le gain décisif : les comparaisons redeviennent possibles

C'est ici que le lissage prouve sa valeur. Reprenons la Partie 7 :

| Phrase | MLE | Laplace |
|---|---|---|
| le chat mange du poisson | 0.055556 | 1.60e-05 |
| le chat mange du pain | **0** | 3.03e-06 |
| le poisson mange du chat | **0** | 6.68e-07 |
| poisson le mange chat du | **0** | 2.38e-08 |

Sans lissage, les trois dernières phrases sont **indistinguables** : toutes à zéro.
Le modèle ne peut rien en dire.

Avec lissage, l'ordre obtenu est exactement le bon :

1. la phrase du corpus ;
2. une phrase correcte contenant un mot inconnu — 5 fois moins probable ;
3. une phrase syntaxiquement correcte mais sémantiquement absurde ;
4. le désordre total — **672 fois** moins probable que S₁.

Le modèle **classe** au lieu d'**annuler**. C'est toute la différence entre un modèle
utilisable et un modèle qui répond zéro à presque tout.

---

## Partie 11 — Comparaison des modèles

Les trois modèles se distinguent par **la quantité de contexte** qu'ils utilisent :

| Modèle | Formule | Contexte | Paramètres |
|---|---|---|---|
| Unigramme | $P(w_i) = \dfrac{C(w_i)}{N}$ | **aucun** | $V$ |
| Bigramme | $P(w_i \mid w_{i-1}) = \dfrac{C(w_{i-1}, w_i)}{C(w_{i-1})}$ | 1 mot | $V^2$ |
| Trigramme | $P(w_i \mid w_{i-2}, w_{i-1}) = \dfrac{C(w_{i-2}, w_{i-1}, w_i)}{C(w_{i-2}, w_{i-1})}$ | 2 mots | $V^3$ |

**Attention au dénominateur du trigramme** : c'est le comptage du **bigramme**
$(w_{i-2}, w_{i-1})$, pas celui d'un unigramme. La règle générale : on divise par le
comptage du contexte, qui compte N−1 mots.

In [121]:
#  Les deux contextes demandés par le TP 
modele.comparer_modeles("le chat")
modele.comparer_modeles("le chat mange")

Contexte : « le chat »
    unigramme (voit aucun contexte        ) : le (0.200)  </s> (0.133)  <s> (0.133)
    bigramme  (voit « chat »              ) : aime (0.333)  joue (0.333)  mange (0.333)
    trigramme (voit « le chat »           ) : aime (0.333)  joue (0.333)  mange (0.333)

Contexte : « le chat mange »
    unigramme (voit aucun contexte        ) : le (0.200)  </s> (0.133)  <s> (0.133)
    bigramme  (voit « mange »             ) : de (0.500)  du (0.500)
    trigramme (voit « chat mange »        ) : du (1.000)



([('le', 0.2), ('</s>', 0.13333333333333333), ('<s>', 0.13333333333333333)],
 [('de', 0.5), ('du', 0.5)],
 [('du', 1.0)])

In [122]:
#  Autres contextes révélateurs 
for contexte in ["chat mange", "joue dans", "mange du", "le chien"]:
    modele.comparer_modeles(contexte)

Contexte : « chat mange »
    unigramme (voit aucun contexte        ) : le (0.200)  </s> (0.133)  <s> (0.133)
    bigramme  (voit « mange »             ) : de (0.500)  du (0.500)
    trigramme (voit « chat mange »        ) : du (1.000)

Contexte : « joue dans »
    unigramme (voit aucun contexte        ) : le (0.200)  </s> (0.133)  <s> (0.133)
    bigramme  (voit « dans »              ) : le (1.000)
    trigramme (voit « joue dans »         ) : le (1.000)

Contexte : « mange du »
    unigramme (voit aucun contexte        ) : le (0.200)  </s> (0.133)  <s> (0.133)
    bigramme  (voit « du »                ) : poisson (1.000)
    trigramme (voit « mange du »          ) : poisson (1.000)

Contexte : « le chien »
    unigramme (voit aucun contexte        ) : le (0.200)  </s> (0.133)  <s> (0.133)
    bigramme  (voit « chien »             ) : aime (0.333)  joue (0.333)  mange (0.333)
    trigramme (voit « le chien »          ) : aime (0.333)  joue (0.333)  mange (0.333)



In [ ]:
#  Q1 et Q2 : quel modèle utilise le moins / le plus de contexte ? 
print(f"{'modèle':12s} {'contexte':>10s} {'distincts':>10s} "
      f"{'possibles':>11s} {'couverture':>11s} {'hapax':>8s}")
print("-" * 68)
for n, nom, compteur in [(1, "unigramme", modele.unigrammes),
                         (2, "bigramme", modele.bigrammes),
                         (3, "trigramme", modele.trigrammes)]:
    possibles = modele.V ** n
    hapax = sum(1 for f in compteur.values() if f == 1)
    print(f"{nom:12s} {n-1:>7d} mot{'s' if n>2 else ' '} "
          f"{len(compteur):>10d} {possibles:>11d} "
          f"{len(compteur)/possibles:>10.2%} {hapax:>4d}/{len(compteur)}")

modèle         contexte  distincts   possibles  couverture    hapax
--------------------------------------------------------------------
unigramme          0 mot          15          15    100.00%    2/15
bigramme           1 mot          23         225     10.22%   13/23
trigramme          2 mots         25        3375      0.74%   19/25


In [ ]:
# Q4 : le trigramme est plus sensible aux comptes nuls 
muets = [(a, b) for a in modele.vocabulaire for b in modele.vocabulaire
         if modele.compte_bigramme(a, b) == 0]

print(f"Contextes de 2 mots possibles : {modele.V ** 2}")
print(f"Contextes jamais observés     : {len(muets)}")
print(f"→ le modèle trigramme est MUET dans "
      f"{len(muets)/modele.V**2:.1%} des cas\n")

print("Exemples de contextes où le trigramme ne peut rien dire :")
for contexte in ["viande poisson", "chat pain", "le mange"]:
    tri = modele.predire_trigramme(contexte)
    bi = modele.predire_mot_suivant(contexte.split()[-1])
    print(f"  « {contexte:16s} » trigramme : {'muet' if not tri else tri[:1]}"
          f"   |   bigramme : {'muet' if not bi else bi[:1]}")

Contextes de 2 mots possibles : 225


Contextes jamais observés     : 202
→ le modèle trigramme est MUET dans 89.8% des cas

Exemples de contextes où le trigramme ne peut rien dire :
  « viande poisson   » trigramme : muet   |   bigramme : [('</s>', 1.0)]
  « chat pain        » trigramme : muet   |   bigramme : muet
  « le mange         » trigramme : muet   |   bigramme : [('de', 0.5)]


In [123]:
#  Mesure objective : la perplexité (cf. §4.6 du cours) 
print("Perplexité (probabilités lissées) — plus c'est BAS, mieux c'est\n")
for phrase in ["le chat mange du poisson",
               "le chat aime la viande",
               "le poisson mange du chat",
               "poisson le mange chat du"]:
    print(f"  {phrase:28s} PP = {modele.perplexite(phrase):>7.2f}")

Perplexité (probabilités lissées) — plus c'est BAS, mieux c'est

  le chat mange du poisson     PP =    6.30
  le chat aime la viande       PP =    5.95
  le poisson mange du chat     PP =   10.70
  poisson le mange chat du     PP =   18.64


### Réponses — Partie 11

**Q1. Quel modèle utilise le moins de contexte ?**

Le **modèle unigramme** : il n'en utilise **aucun**. $P(w_i) = C(w_i)/N$ ne dépend que
de la fréquence brute du mot.

La conséquence est visible dans les résultats : quel que soit le contexte — « le chat »,
« le chat mange », « joue dans » — il prédit **toujours la même chose** : `le` (0.200),
le mot le plus fréquent du corpus. Il ne « prédit » rien, il récite un classement.

Un texte généré par un unigramme est un sac de mots tirés selon leurs fréquences :
*« le le poisson dans le chat »*. Aucune structure.

**Q2. Quel modèle utilise le plus de contexte ?**

Le **modèle trigramme** : deux mots précédents. Sur « le chat mange », il conditionne
sur `chat mange` là où le bigramme ne voit que `mange`.

**Q3. Pourquoi le trigramme est-il généralement plus précis ?**

Parce qu'un contexte plus long est **plus discriminant**. La comparaison sur
« le chat mange » le montre exactement :

- Le bigramme ne voit que `mange`. Or `mange` est suivi de `du` (dans « le chat mange
  **du** poisson ») et de `de` (dans « le chien mange **de** la viande »). Il répond
  donc `de` 0.5 / `du` 0.5 — **incapable de trancher**.
- Le trigramme voit `chat mange`. Le corpus n'a associé cette séquence qu'à `du`. Il
  répond `du` avec **certitude**.

L'information qui manquait au bigramme — le sujet est un chat, donc du poisson — était
présente dans le mot d'avant. Le trigramme la capte, le bigramme l'a oubliée.

Autrement dit : plus le contexte est long, plus la distribution conditionnelle est
**concentrée**, donc plus l'entropie est faible et la prédiction fiable. C'est ce que
mesure la perplexité.

**Q4. Pourquoi le trigramme est-il aussi plus sensible aux comptes nuls ?**

Parce que le nombre de paramètres à estimer croît **exponentiellement** avec N, alors
que le corpus, lui, ne grandit pas :

| Modèle | Paramètres à estimer | Observés | Couverture |
|---|---|---|---|
| unigramme | $V$ = 15 | 15 | **100 %** |
| bigramme | $V^2$ = 225 | 23 | 10.2 % |
| trigramme | $V^3$ = 3 375 | 25 | **0.74 %** |

Passer du bigramme au trigramme multiplie l'espace des paramètres par 15, tandis que le
nombre d'observations **diminue** (33 trigrammes contre 39 bigrammes, car chaque phrase
en fournit un de moins).

Deux conséquences mesurées :

- **Le modèle devient muet.** Il ne peut prédire que si le bigramme de contexte a été
  observé. Ici, 202 contextes sur 225 sont inconnus : **le trigramme ne peut rien dire
  dans 90 % des cas**. Le bigramme, lui, répond dès que le mot précédent est dans le
  vocabulaire.
- **Les estimations reposent sur presque rien.** 19 trigrammes sur 25 sont des hapax
  (76 %), contre 13 sur 23 pour les bigrammes (57 %). Une probabilité de 1.000 fondée
  sur une observation unique n'a aucune valeur statistique.

**C'est l'arbitrage biais-variance appliqué aux modèles de langage.** Un N faible donne
un modèle grossier mais bien estimé (biais élevé, variance faible) ; un N élevé donne un
modèle fin mais mal estimé (biais faible, variance élevée). L'optimum dépend de la taille
du corpus, et se situe en pratique autour de N = 3 à 5 sur de très grands corpus.

En production, on ne choisit d'ailleurs pas : on **combine**. Le *backoff* de Katz recule
vers le bigramme quand le trigramme est absent, puis vers l'unigramme. L'interpolation
mélange les trois avec des poids appris. On garde ainsi la précision du trigramme quand
les données existent, et la robustesse du bigramme sinon.

**Q5. Que se passe-t-il lorsque la taille du corpus augmente ?**

Quatre effets :

1. **Les estimations se fiabilisent.** Chaque probabilité repose sur davantage
   d'observations ; la variance diminue. $P(\text{poisson} \mid \text{du}) = 1/1$ — une
   certitude illusoire ici — deviendrait une valeur nuancée.
2. **La couverture s'améliore, mais lentement.** Les N-grammes fréquents apparaissent
   vite ; la longue traîne, jamais. Par la **loi de Zipf**, une part importante du
   vocabulaire reste rare quelle que soit la taille du corpus.
3. **Les modèles d'ordre élevé deviennent praticables.** C'est la condition d'existence
   du trigramme : il n'est exploitable qu'à partir de plusieurs millions de tokens. Sur
   45 tokens, il est inutilisable.
4. **Le problème des comptes nuls ne disparaît jamais.** Avec V = 100 000, il y a
   $10^{15}$ trigrammes possibles — aucun corpus n'en couvrira une fraction
   significative. **Le lissage reste indispensable à toute échelle.**

C'est la raison profonde pour laquelle les modèles N-grammes ont été remplacés : leur
besoin en données croît exponentiellement avec la longueur du contexte, alors que les
modèles neuronaux représentent les mots dans un espace continu et **généralisent** aux
séquences non vues. C'est l'objet du défi final.

### La perplexité : une mesure objective

Comparer les modèles « à l'œil » sur quelques contextes ne suffit pas. La mesure standard
est la **perplexité** (§4.6 du cours) :

$$PP(S) = 2^{-\frac{1}{n}\log_2 P(S)}$$

Elle s'interprète comme le **nombre moyen de choix équiprobables** auxquels le modèle
fait face à chaque mot. PP = 6 signifie « le modèle hésite comme s'il tirait entre
6 possibilités ». Plus la perplexité est basse, meilleur est le modèle.

| Phrase | PP |
|---|---|
| le chat mange du poisson | **6.30** |
| le chat aime la viande | 6.30 |
| le poisson mange du chat | 12.83 |
| poisson le mange chat du | **18.64** |

L'ordre reproduit exactement la plausibilité linguistique. La normalisation par la
longueur permet en outre de comparer des phrases de tailles différentes — ce que la
probabilité brute ne permettait pas (Partie 6).

---

## Partie 12 — Mini-projet

Le programme interactif est dans le fichier **`mini_modele_langage.py`**, à la racine du
projet. Il propose les 11 options demandées par le TP et réutilise les fonctions de
`modele_langage.py` — aucune logique n'y est dupliquée.

```
=========================================
        MINI MODELE DE LANGAGE
=========================================
 1. Afficher le vocabulaire
 2. Afficher les unigrammes
 3. Afficher les bigrammes
 4. Afficher les trigrammes
 5. Calculer une probabilite
 6. Predire le mot suivant
 7. Generer une phrase
 8. Calculer la probabilite d'une phrase
 9. Corriger une phrase
10. Comparer deux phrases
11. Quitter
=========================================
```

Lancement, depuis la racine du projet :

```bash
python mini_modele_langage.py
```

Trois choix de conception :

- **Séparation interface / logique.** Le fichier ne contient que la saisie, l'affichage
  et l'aiguillage. C'est ce qui permet au notebook et au programme de partager
  exactement le même code.
- **Deux modèles en mémoire.** Le corpus principal pour les options 1 à 8 et 10, le
  corpus de correction pour l'option 9 — impossible avec des variables globales.
- **Valeurs par défaut partout.** Appuyer sur Entrée reprend l'exemple du TP, ce qui
  permet de démontrer les 10 options sans rien taper.

---

# Questions de synthèse

**1. Qu'est-ce qu'un modèle de langage ?**

Une fonction qui attribue une **probabilité** à toute séquence de mots, et par
conséquent estime la probabilité qu'un mot apparaisse après un contexte donné. Ce n'est
pas une grammaire : il ne dit pas si une phrase est *correcte*, mais à quel point elle
est **attendue** au vu d'un corpus. Deux usages symétriques en découlent : **évaluer**
une séquence existante (Parties 6 à 8) et **générer** une séquence nouvelle (Partie 5).

**2. Quelle est la différence entre un corpus et un vocabulaire ?**

Le **corpus** est l'ensemble des textes observés — ici 6 phrases, N = 45 tokens. Le
**vocabulaire** est l'ensemble des mots *distincts* qui y apparaissent — V = 15.

Le corpus contient des **occurrences** (le mot `le` y figure 9 fois), le vocabulaire des
**types** (`le` y figure une fois). Le corpus est la donnée d'entraînement ; le
vocabulaire est l'espace des sorties possibles du modèle. C'est V, et non N, qui
apparaît au dénominateur du lissage de Laplace.

**3. Quelle est la différence entre un unigramme, un bigramme et un trigramme ?**

Le nombre de tokens dans la séquence, donc la quantité de contexte utilisée :

| | Séquence | Contexte | Formule |
|---|---|---|---|
| Unigramme | 1 mot | aucun | $P(w_i)$ |
| Bigramme | 2 mots | 1 mot | $P(w_i \mid w_{i-1})$ |
| Trigramme | 3 mots | 2 mots | $P(w_i \mid w_{i-2}, w_{i-1})$ |

**4. Pourquoi un modèle bigramme utilise-t-il une probabilité conditionnelle ?**

Parce que son objet est précisément de modéliser une **dépendance** : la probabilité
d'un mot n'est pas absolue, elle dépend de ce qui précède. Une probabilité simple
$P(\text{chat})$ ne dirait que la fréquence globale du mot ; $P(\text{chat} \mid \text{le})$
dit comment le contexte modifie cette attente.

C'est ce qui distingue le bigramme de l'unigramme, lequel n'utilise que des probabilités
non conditionnelles et se comporte comme un sac de mots.

**5. Que signifie $P(\text{chat} \mid \text{le})$ ?**

« La probabilité que le mot suivant soit `chat`, **sachant** que le mot précédent est
`le`. » Estimée par $C(\text{le}, \text{chat}) / C(\text{le}) = 3/9 = 0.333$ : sur les
9 occurrences de `le` dans le corpus, 3 sont suivies de `chat`.

**6. Pourquoi $P(\text{chat} \mid \text{le}) \neq P(\text{le} \mid \text{chat})$ en général ?**

Parce que les deux quantités ne comparent pas les mêmes comptages :

$$P(\text{chat} \mid \text{le}) = \frac{C(\text{le}, \text{chat})}{C(\text{le})} = \frac{3}{9} \qquad
P(\text{le} \mid \text{chat}) = \frac{C(\text{chat}, \text{le})}{C(\text{chat})} = \frac{0}{3}$$

Les dénominateurs diffèrent (9 contre 3) et les numérateurs aussi, car **un bigramme est
ordonné** : $(le, chat)$ et $(chat, le)$ sont deux N-grammes distincts. Par Bayes,
$P(A \mid B) = P(B \mid A)P(A)/P(B)$ : l'égalité n'aurait lieu que si $P(A) = P(B)$.

**7. Comment un modèle N-gramme peut-il prédire le mot suivant ?**

En cherchant, parmi les successeurs observés du contexte, celui de plus forte
probabilité :

$$\hat{w} = \arg\max_{w \in V} P(w \mid \text{contexte})$$

Concrètement : filtrer les N-grammes commençant par le contexte, diviser leurs
fréquences par celle du contexte, prendre le maximum.

**8. Comment peut-il générer une phrase ?**

En itérant la prédiction : partir de `<s>`, prédire le mot suivant, l'ajouter au
contexte, recommencer jusqu'à produire `</s>`. Le choix à chaque étape peut être
l'argmax (déterministe, une seule phrase possible) ou un **tirage aléatoire selon la
distribution** (varié) — c'est la différence entre température 0 et température 1 dans
les modèles modernes.

**9. Comment peut-il comparer deux phrases ?**

En calculant P(S) pour chacune par la règle de la chaîne et en comparant. La phrase de
plus forte probabilité est la plus conforme aux régularités du corpus. En pratique, on
compare des **log-probabilités** (pour éviter le soupassement) et, si les longueurs
diffèrent, des **perplexités** (qui normalisent).

**10. Pourquoi les comptes nuls constituent-ils un problème ?**

Parce que P(S) est un **produit** : un seul facteur nul l'annule entièrement. Une phrase
correcte de 20 mots reçoit P = 0 à cause d'un unique bigramme jamais rencontré. Le
modèle perd alors tout pouvoir de discrimination — une phrase correcte et une phrase
absurde sont toutes deux à zéro, donc indistinguables.

Le problème est massif (202 bigrammes nuls sur 225 ici) et **structurel** : par la loi
de Zipf, aucun corpus, si grand soit-il, ne couvre l'espace des N-grammes possibles.

**11. Quel est le rôle du lissage de Laplace ?**

Redistribuer une petite part de la masse de probabilité des événements observés vers les
événements non observés, pour qu'aucune probabilité ne soit nulle :

$$P_{\text{Laplace}}(w_i \mid w_{i-1}) = \frac{C(w_{i-1}, w_i) + 1}{C(w_{i-1}) + V}$$

Le +1 supprime les zéros, le +V préserve la normalisation. Le modèle passe d'un verdict
binaire (« impossible ») à un **classement gradué**, ce qui rend les comparaisons de
phrases à nouveau possibles.

**12. Pourquoi un trigramme peut-il être plus performant qu'un bigramme ?**

Parce qu'un contexte de deux mots est plus discriminant. Exemple mesuré dans ce TP :
après « le chat mange », le bigramme ne voit que `mange` et hésite entre `de` (0.5) et
`du` (0.5) ; le trigramme voit `chat mange` et répond `du` avec certitude. L'information
utile — le sujet est un chat — était dans le mot que le bigramme avait oublié.

**Mais ce gain n'est pas gratuit** : l'espace des paramètres passe de $V^2$ à $V^3$
tandis que les observations diminuent. Sur ce corpus, le trigramme est muet dans 90 %
des contextes. Le trigramme n'est supérieur qu'à partir d'un corpus suffisamment grand.

**13. Pourquoi les modèles N-grammes ont-ils des limites lorsqu'on travaille avec des textes longs ?**

Quatre limites, toutes visibles dans ce TP :

- **Contexte borné.** Un modèle d'ordre N ne voit que N−1 mots. Toute dépendance plus
  longue — accord sujet-verbe à distance, référence anaphorique, cohérence d'un
  paragraphe — lui est invisible. C'est ce qui produit *« le chat joue dans le chien
  aime la viande »* : chaque transition est valide, l'ensemble ne l'est pas.
- **Explosion combinatoire.** Le nombre de paramètres croît en $V^N$. Avec V = 100 000
  et N = 5, il y a $10^{25}$ paramètres — ni estimables, ni stockables.
- **Aucune généralisation.** Les mots sont des symboles discrets sans relation entre
  eux : `chat` et `chien` sont aussi étrangers l'un à l'autre que `chat` et `jardin`.
  Avoir vu « le chat mange » n'aide en rien à évaluer « le chien mange ».
- **Comptes nuls persistants.** Plus le texte est long, plus il contient de N-grammes
  jamais vus, et plus la probabilité de l'ensemble tend vers zéro.

---

# Défi supplémentaire — Vers les modèles de langage modernes

**Pourquoi les modèles comme GPT n'utilisent-ils pas simplement des N-grammes ?**

Chacune des six limites listées par le TP a été rencontrée concrètement dans ce travail.

### 1. Contexte limité

Un bigramme voit un mot ; un trigramme, deux. Impossible d'aller beaucoup plus loin sans
faire exploser le nombre de paramètres.

Un Transformer traite **le contexte entier en une seule fois** grâce au mécanisme
d'**attention** : chaque mot peut se relier directement à n'importe quel autre mot de la
séquence, quelle que soit la distance. Les fenêtres de contexte actuelles atteignent des
centaines de milliers de tokens, là où le N-gramme plafonne à 4 ou 5.

### 2. Explosion du nombre de N-grammes

Mesuré ici : $V$ = 15 → 15 paramètres, $V^2$ = 225, $V^3$ = 3 375. La croissance est
exponentielle en N, et le corpus, lui, ne suit pas.

Un réseau neuronal a un nombre de paramètres **fixe**, indépendant de la longueur du
contexte. Ajouter dix mots de contexte ne change pas la taille du modèle — seulement le
calcul effectué. C'est ce qui débloque les contextes longs.

### 3. Problèmes de généralisation

C'est la limite la plus profonde. Pour un modèle N-gramme, les mots sont des **symboles
discrets** : `chat` et `chien` n'ont aucune parenté, pas plus que `chat` et `jardin`.
Notre corpus a vu « le chien mange de la viande », ce qui n'aide **en rien** à évaluer
« le chat mange de la viande ».

Les modèles neuronaux représentent chaque mot par un **vecteur dense** (*embedding*)
dans un espace continu où les mots de sens proche sont géométriquement proches.
`chat` et `chien` y sont voisins, car ils apparaissent dans des contextes similaires.
Le modèle **transfère** alors ce qu'il a appris de l'un vers l'autre : il peut évaluer
correctement une séquence qu'il n'a jamais vue.

C'est le saut conceptuel décisif : passer de la **mémorisation de séquences** à
l'**apprentissage de représentations**.

### 4. Comptes nuls

Nous avons mesuré 202 bigrammes nuls sur 225, soit 90 %. Le lissage atténue le symptôme
mais reste arbitraire : il attribue la **même** probabilité à `(chat, pain)` et à
`(viande, poisson)`, alors que la première séquence est plausible et la seconde non.

Un modèle neuronal n'a pas de comptes du tout. Il produit une distribution continue sur
tout le vocabulaire, calculée par une fonction apprise. La notion même de « séquence
jamais vue, donc probabilité nulle » disparaît : **toute** séquence reçoit une
probabilité, calculée par similarité avec ce qui a été appris.

### 5. Dépendances longues

Le correcteur de la Partie 8 ne peut pas traiter « les enfants que j'ai vu » → « vus » :
le mot déclencheur est cinq positions en amont, hors de portée d'un bigramme.

L'attention résout exactement ce problème : le modèle apprend **quels mots regarder**
pour prédire le suivant, sans contrainte de distance. Un pronom peut se lier à son
antécédent trois phrases plus haut.

### 6. Absence de compréhension du contexte

Un N-gramme ne manipule que des fréquences. Il ne « sait » pas qu'un chat est un animal,
qu'un animal mange, que manger prend un complément. Toute la structure du langage lui
reste extérieure.

Les modèles modernes apprennent, à travers des milliards d'exemples et des couches de
représentation successives, des régularités qui s'apparentent à de la syntaxe et de la
sémantique — sans qu'aucune règle n'ait jamais été programmée.

### Ce que les N-grammes conservent

Il serait faux d'en conclure qu'ils sont obsolètes. Ils gardent des atouts réels :

- **Transparence totale.** Chaque probabilité se ramène à une division entre deux
  comptages. On peut auditer n'importe quelle décision — ce qu'aucun LLM ne permet.
- **Coût négligeable.** Notre modèle s'entraîne instantanément sur un ordinateur
  portable ; GPT-4 a demandé des mois de calcul sur des milliers de GPU.
- **Usages actuels.** Ils restent employés en reconnaissance vocale embarquée, en
  claviers prédictifs, en détection de langue, et comme **base de comparaison** pour
  évaluer des modèles plus lourds.

### Le fil conducteur

Et surtout, les concepts établis dans ce TP n'ont pas disparu — ils sont ceux des
modèles modernes :

| Concept N-gramme | Équivalent moderne |
|---|---|
| $P(w_i \mid \text{contexte})$ | objectif d'entraînement identique |
| argmax vs échantillonnage | température, top-k, top-p |
| log-probabilités | log-vraisemblance, fonction de perte |
| perplexité | métrique d'évaluation standard |
| lissage | régularisation, *label smoothing* |
| `<s>` et `</s>` | tokens spéciaux `<BOS>`, `<EOS>` |

GPT résout la **même équation** que notre modèle bigramme — prédire le mot suivant
sachant le contexte. Il ne change pas l'objectif : il change la manière de l'estimer,
en remplaçant le comptage de séquences par l'apprentissage de représentations continues.

---

## Conclusion

Le cheminement suivi :

```
Corpus → Tokenisation → N-grammes → Fréquences → Probabilités → Modèle de langage
                                                                      ↓
                              Prédiction · Génération · Correction · Évaluation
```

Un modèle N-gramme transforme des **observations** issues d'un corpus en
**probabilités**, puis utilise ces probabilités pour prendre des **décisions** sur des
séquences nouvelles. Toute la chaîne tient dans cette phrase — et c'est encore, à une
autre échelle, ce que font les modèles de langage d'aujourd'hui.